# Baseline LSTM v4 — Calibrated Confidence + Dynamic Risk Budget

**Author:** Minho  
**Project:** Risk-Adjusted Portfolio Optimization  
**Model direction:** Ranking LSTM with calibrated confidence, explicit downside prediction, dynamic concentration, and risk-aware portfolio sizing  
**Dataset:** configurable through `DATASET_NAME` / `dataset`; designed to transfer to the hidden biotech-style universe  

Leader update incorporated: v3 found winners, but the allocator stayed close to a fixed broad basket. v4 makes confidence change active-name count, max-weight budget, risk penalties, and loser exclusion so the portfolio becomes broad/defensive when uncertain and selective only when alpha confidence and risk quality agree.


## 0. Bootstrap — locate repo root

In [ ]:
import os
import sys
from pathlib import Path

# --- If auto-detection fails, set this manually ---
# repo_root = Path('/Users/minhochoi/Portfolio-Optimization-Lib') 
# --------------------------------------------------

def _is_repo_root(p: Path) -> bool:
    return (p / 'pyproject.toml').exists() and (p / 'src' / 'portfolio_toolkit').exists()

if 'repo_root' not in dir() or not _is_repo_root(Path(repo_root)):
    _candidates = [Path.cwd(), *Path.cwd().parents]
    _found = next((p for p in _candidates if _is_repo_root(p)), None)
    if _found is None:
        raise RuntimeError(
            'Cannot find repo root. Uncomment and set repo_root manually at the top of this cell.'
        )
    repo_root = _found

repo_root = Path(repo_root).resolve()
os.chdir(repo_root)

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('repo_root:', repo_root)
print('python:   ', sys.executable)

## 1. Imports

In [ ]:
import random
import warnings

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from portfolio_toolkit import (
    backtest_weights,
    build_features,
    get_dataset_spec,
    custom_dataset,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    slice_split,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_rank_long_only,
    write_backtest_artifacts,
)

warnings.filterwarnings('ignore')
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

--------------------------------------------------------------------------------

### `build_model_features(prices)` & `predict_from_prices(model, prices, dates=None, tickers=None)`

In [ ]:
# Reusable model functions

def _add_market_regime_features(prices: pd.DataFrame, frame: pd.DataFrame) -> pd.DataFrame:
    """Attach date-level benchmark, breadth, dispersion, and regime context."""
    panel = prices.sort_values(['ticker', 'date']).copy()
    panel['ret_1d'] = panel.groupby('ticker')['adj_close'].pct_change()
    panel['ret_5d'] = panel.groupby('ticker')['adj_close'].pct_change(5)
    panel['ret_20d'] = panel.groupby('ticker')['adj_close'].pct_change(20)

    spy = panel.loc[panel['ticker'] == 'SPY', ['date', 'adj_close', 'ret_1d', 'ret_5d', 'ret_20d']].copy()
    spy = spy.sort_values('date')
    spy['spy_vol_20d'] = spy['ret_1d'].rolling(20, min_periods=20).std(ddof=0)
    spy['spy_vol_60d'] = spy['ret_1d'].rolling(60, min_periods=60).std(ddof=0)
    spy['spy_sma_20d'] = spy['adj_close'].rolling(20, min_periods=20).mean()
    spy['spy_sma_60d'] = spy['adj_close'].rolling(60, min_periods=60).mean()
    spy['spy_trend_20d'] = spy['adj_close'] / spy['spy_sma_20d'] - 1.0
    spy['spy_trend_60d'] = spy['adj_close'] / spy['spy_sma_60d'] - 1.0
    spy['spy_drawdown_60d'] = spy['adj_close'] / spy['adj_close'].rolling(60, min_periods=20).max() - 1.0
    spy['spy_rebound_5d'] = spy['ret_5d'].clip(lower=0.0)

    tradable_panel = panel.loc[panel['ticker'] != 'SPY'].copy()
    breadth = tradable_panel.groupby('date').agg(
        universe_breadth_20d=('ret_20d', lambda s: float((s > 0).mean())),
        universe_dispersion_5d=('ret_5d', 'std'),
        universe_dispersion_20d=('ret_20d', 'std'),
        universe_median_return_5d=('ret_5d', 'median'),
        universe_median_return_20d=('ret_20d', 'median'),
    ).reset_index()

    regime = spy[[
        'date', 'ret_5d', 'ret_20d', 'spy_vol_20d', 'spy_vol_60d',
        'spy_trend_20d', 'spy_trend_60d', 'spy_drawdown_60d', 'spy_rebound_5d'
    ]].rename(columns={'ret_5d': 'spy_return_5d', 'ret_20d': 'spy_return_20d'})
    regime = regime.merge(breadth, on='date', how='left')
    regime['high_vol_regime'] = (regime['spy_vol_20d'] >= regime['spy_vol_20d'].rolling(252, min_periods=60).median()).astype(float)
    regime['drawdown_regime'] = (regime['spy_drawdown_60d'] <= -0.08).astype(float)
    regime['rebound_regime'] = ((regime['spy_return_5d'] > 0.03) & (regime['spy_drawdown_60d'] > -0.12)).astype(float)

    return frame.merge(regime, on='date', how='left')


def _add_cross_sectional_features(frame: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    """Add within-date ranks and z-scores so the LSTM sees relative opportunity."""
    out = frame.copy()
    grouped = out.groupby('date', sort=False)
    for col in base_cols:
        if col not in out.columns:
            continue
        out[f'{col}_cs_rank'] = grouped[col].rank(pct=True, method='average')
        mean = grouped[col].transform('mean')
        std = grouped[col].transform('std').replace(0.0, np.nan)
        out[f'{col}_cs_z'] = (out[col] - mean) / std
    return out


def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    """
    Build the full feature matrix from raw prices.

    v4 adds cross-sectional ranks/z-scores and market-regime context so the
    sequence model can distinguish ordinary momentum from unusually strong
    relative opportunity in calm, drawdown, and rebound regimes.
    """
    base_features = [
        'return_1d', 'return_5d', 'return_10d', 'return_20d', 'return_60d',
        'vol_5d', 'vol_10d', 'vol_20d', 'vol_60d',
        'downside_vol_20d', 'upside_vol_20d', 'atr_14',
        'momentum_5d', 'momentum_10d', 'momentum_20d', 'momentum_60d', 'momentum_120d',
        'rsi_14', 'macd_hist', 'bollinger_z_20d',
        'beta_20d_spy', 'beta_60d_spy',
        'excess_return_5d_vs_spy', 'excess_return_20d_vs_spy', 'excess_return_60d_vs_spy',
        'relative_momentum_20d_vs_spy',
        'volume_zscore_20d', 'volume_zscore_60d', 'dollar_volume_ratio_20d',
        'price_to_sma_20d', 'price_to_sma_50d', 'price_to_sma_200d',
        'intraday_range', 'close_open_gap', 'close_location_in_range',
        'distance_to_20d_high', 'distance_to_20d_low',
        'distance_to_60d_high', 'distance_to_60d_low',
        'skew_20d', 'kurtosis_20d',
    ]

    feature_frame = build_features(prices, feature_names=base_features)

    panel = prices.sort_values(['ticker', 'date']).copy()
    ret5  = panel.groupby('ticker')['adj_close'].pct_change(5)
    ret20 = panel.groupby('ticker')['adj_close'].pct_change(20)
    ret60 = panel.groupby('ticker')['adj_close'].pct_change(60)

    custom_df = panel[['date', 'ticker']].copy()
    custom_df['price_accel'] = (ret5 - ret20).values
    custom_df['momentum_reversal_20_60'] = (ret20 - ret60).values
    custom_df['vol_compression_20_60'] = (
        panel.groupby('ticker')['adj_close'].pct_change()
        .groupby(panel['ticker']).transform(lambda s: s.rolling(20, min_periods=20).std(ddof=0))
        /
        panel.groupby('ticker')['adj_close'].pct_change()
        .groupby(panel['ticker']).transform(lambda s: s.rolling(60, min_periods=60).std(ddof=0)).replace(0.0, np.nan)
    ).values

    frame = feature_frame.merge(custom_df, on=['date', 'ticker'], how='left')
    frame = _add_market_regime_features(prices, frame)

    cs_cols = [
        'momentum_20d', 'momentum_60d', 'excess_return_20d_vs_spy',
        'relative_momentum_20d_vs_spy', 'vol_20d', 'downside_vol_20d',
        'volume_zscore_20d', 'price_accel', 'distance_to_20d_high',
    ]
    frame = _add_cross_sectional_features(frame, cs_cols)
    return frame


def _model_outputs(model: nn.Module, X_tensor: torch.Tensor):
    outputs = model(X_tensor)
    if isinstance(outputs, dict):
        return outputs
    return {'alpha': outputs}


def predict_from_prices(
    model: nn.Module,
    prices: pd.DataFrame,
    train_mean: pd.Series,
    train_std: pd.Series,
    feature_names: list,
    seq_len: int,
    horizon: int,
    dates=None,
    tickers=None,
) -> pd.DataFrame:
    """Generate the standard prediction frame from raw prices."""
    frame = build_model_features(prices)
    frame = frame.replace([np.inf, -np.inf], np.nan).dropna(subset=feature_names)

    X_scaled = ((frame[feature_names] - train_mean) / train_std).to_numpy(dtype=np.float32)

    X_seqs, meta = [], []
    frame = frame.reset_index(drop=True)
    for ticker, grp in frame.groupby('ticker', sort=False):
        idx = grp.index.tolist()
        dates_grp = grp['date'].tolist()
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx)):
            rows = idx[end - seq_len + 1 : end + 1]
            X_seqs.append(X_scaled[rows])
            meta.append((dates_grp[end], ticker))

    if not X_seqs:
        raise ValueError('No sequences could be built; not enough rows per ticker.')

    model.eval()
    device = next(model.parameters()).device
    alpha_list, confidence_list, bottom_list, vol_list, downside_list, worst_path_list = [], [], [], [], [], []
    aux_lists = {h: [] for h in globals().get('AUX_HORIZONS', [])}
    with torch.no_grad():
        for i in range(0, len(X_seqs), 2048):
            X_tensor = torch.as_tensor(np.stack(X_seqs[i:i+2048]), dtype=torch.float32, device=device)
            outputs = _model_outputs(model, X_tensor)
            alpha_list.append(outputs['alpha'].detach().cpu().numpy())
            if 'confidence_logit' in outputs:
                confidence_list.append(torch.sigmoid(outputs['confidence_logit']).detach().cpu().numpy())
            if 'bottom_logit' in outputs:
                bottom_list.append(torch.sigmoid(outputs['bottom_logit']).detach().cpu().numpy())
            if 'volatility' in outputs:
                vol_list.append(torch.nn.functional.softplus(outputs['volatility']).detach().cpu().numpy())
            if 'downside' in outputs:
                downside_list.append(torch.nn.functional.softplus(outputs['downside']).detach().cpu().numpy())
            if 'worst_path' in outputs:
                worst_path_list.append((-torch.nn.functional.softplus(outputs['worst_path'])).detach().cpu().numpy())
            for h in aux_lists:
                key = f'alpha_{h}d'
                if key in outputs:
                    aux_lists[h].append(outputs[key].detach().cpu().numpy())

    pred_dates, pred_tickers = zip(*meta)
    predictions = pd.DataFrame({
        'date': pd.to_datetime(list(pred_dates)),
        'ticker': list(pred_tickers),
        'horizon': horizon,
        'expected_return': np.concatenate(alpha_list).astype(float),
    })
    predictions['expected_alpha'] = predictions['expected_return']
    if confidence_list:
        predictions['confidence'] = np.concatenate(confidence_list).astype(float)
        predictions['uncertainty'] = 1.0 - predictions['confidence']
    else:
        predictions['confidence'] = 0.5
        predictions['uncertainty'] = 0.5
    if bottom_list:
        predictions['bottom_probability'] = np.concatenate(bottom_list).astype(float)
    else:
        predictions['bottom_probability'] = 0.2
    if vol_list:
        predictions['expected_volatility'] = np.concatenate(vol_list).astype(float)
    if downside_list:
        predictions['expected_downside'] = np.concatenate(downside_list).astype(float)
    if worst_path_list:
        predictions['expected_worst_path'] = np.concatenate(worst_path_list).astype(float)
    for h, values in aux_lists.items():
        if values:
            predictions[f'expected_alpha_{h}d'] = np.concatenate(values).astype(float)

    predictions = predictions.drop_duplicates(['date', 'ticker', 'horizon'], keep='last')

    if tickers is not None:
        predictions = predictions[predictions['ticker'].isin(tickers)]
    if dates is not None:
        predictions = predictions[predictions['date'].isin(pd.to_datetime(dates))]

    return predictions.sort_values(['date', 'ticker']).reset_index(drop=True)


print('v4 feature builder and prediction helper defined.')


----------------------------------------------------------------------------------

## 2. Config

v4 keeps the LSTM ranking backbone, but changes the experiment around the leader's feedback: confidence must change the portfolio, not merely appear in diagnostics. The config below exposes dynamic top-N bands, confidence/spread/risk gates, loser filters, multi-horizon auxiliary targets, no-trade bands, and cap schedules.


### Retired baseline config

The original `shared_set_1` config has been retired from the executable path. The active config below is the v4 calibrated-confidence setup. Keeping this note avoids accidentally running the stale baseline cell with the invalid device branch and old run name.


### Active v4 experiment config

Use this cell to choose the local training universe and the confidence/risk allocator behavior. The hidden review target is a broad, volatile innovation universe, so the portfolio is allowed to concentrate only when alpha confidence, score spread, and risk quality are all acceptable.


In [ ]:
from portfolio_toolkit import custom_dataset
from dataclasses import replace
from datetime import date

TRAINING_CUTOFF = date(2022, 12, 31)

# Local development universe. Hidden evaluation can replace this dataset while reusing
# the same feature, model, prediction, and portfolio code.
SUBSET_TICKERS = [
    # Technology
    'AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'AVGO', 'ORCL', 'ADBE', 'CRM', 'AMD', 'INTC',
    # Financials
    'JPM', 'BAC', 'GS', 'MS', 'BLK', 'SPGI', 'V', 'MA',
    # Healthcare / biotech-adjacent large caps
    'LLY', 'UNH', 'JNJ', 'ABBV', 'MRK', 'TMO',
    # Consumer
    'AMZN', 'TSLA', 'HD', 'MCD', 'NKE', 'SBUX',
    # Industrials
    'CAT', 'DE', 'UPS', 'HON', 'GE', 'RTX',
    # Energy
    'XOM', 'CVX', 'COP', 'SLB',
    # Communication
    'DIS', 'NFLX', 'CMCSA', 'T',
    # Utilities / defensives
    'PG', 'KO', 'PEP', 'WMT', 'COST',
]

dataset = custom_dataset(
    tickers   = SUBSET_TICKERS,
    start     = '2014-01-02',
    end       = '2025-12-31',
    benchmark = 'SPY',
    name      = 'sp500_subset_50_v4_dev',
)

dataset = replace(
    dataset,
    train_start = date(2014, 1,  2),
    train_end   = date(2021, 12, 31),
    val_start   = date(2022, 1,  3),
    val_end     = TRAINING_CUTOFF,
    test_start  = date(2023, 1,  3),
    test_end    = date(2025, 12, 31),
)

# Core experiment identity
DATASET_NAME     = dataset
RUN_NAME         = 'Minho_RankingConfidence_LSTM_v4_dynamic_risk'
MODEL_FAMILY     = 'ranking_confidence_lstm'
TARGET_TYPE      = 'alpha_rank_confidence_downside_multihorizon'

# Forecasting setup
HORIZON          = 5
AUX_HORIZONS     = [10, 20]
SEQ_LEN          = 20
SEQ_LEN_SWEEP    = [10, 20, 40, 60]
MODEL_SWEEP      = ['lstm', 'mlp_baseline']  # compare later to verify the sequence model adds value
ALPHA_BENCHMARK_TICKER = 'SPY'

# Model hyperparameters
HIDDEN_SIZE      = 64
NUM_LAYERS       = 2
DROPOUT          = 0.2
LR               = 1e-3
EPOCHS           = 30
BATCH_SIZE       = 512
SEED             = 42

# Multi-task loss weights
ALPHA_LOSS_WEIGHT       = 1.00
RANK_LOSS_WEIGHT        = 0.35
TOP_QUINTILE_WEIGHT     = 0.30
BOTTOM_QUINTILE_WEIGHT  = 0.25
VOL_LOSS_WEIGHT         = 0.15
DOWNSIDE_LOSS_WEIGHT    = 0.25
WORST_PATH_LOSS_WEIGHT  = 0.20
AUX_ALPHA_LOSS_WEIGHT   = 0.20

# Portfolio experiment knobs: confidence changes active risk, not just diagnostics.
REBALANCE_FREQ          = 5
REBALANCE_FREQ_SWEEP    = [5, 10, 21]
DYNAMIC_TOP_N           = {'high': 15, 'normal': 28, 'low': 45}
SLEEVE_SIZES            = [12, 18, 29, 45]
CONFIDENCE_THRESHOLD    = 0.62
HIGH_CONFIDENCE_THRESHOLD = 0.70
LOW_CONFIDENCE_THRESHOLD  = 0.54
SPREAD_THRESHOLD        = 0.015
RISK_QUALITY_THRESHOLD  = 0.55
SOFTMAX_TEMPERATURE     = 0.20
DEFENSIVE_TEMPERATURE   = 0.45
RISK_AVERSION           = 1.75
DOWNSIDE_AVERSION       = 2.25
TAIL_RISK_AVERSION      = 1.75
MAX_WEIGHT              = 0.040
DEFENSIVE_MAX_WEIGHT    = 0.025
RISK_ON_MAX_WEIGHT      = 0.060
MIN_ACTIVE_NAMES        = 12
MIN_EFFECTIVE_NAMES     = 10
LOSER_RANK_CUTOFF       = 0.12
BOTTOM_PROB_CUTOFF      = 0.62
NO_TRADE_BAND           = 0.0050
TARGET_BETA             = None

artifact_dir     = repo_root / 'outputs' / RUN_NAME
artifact_dir.mkdir(parents=True, exist_ok=True)

spec             = dataset
BENCHMARK_TICKER = spec.benchmark_ticker
TRADABLE_TICKERS = [t for t in spec.tickers if t != BENCHMARK_TICKER]

assert spec.train_end <= TRAINING_CUTOFF, 'Train split leaks beyond the Dec 2022 cutoff'
assert spec.val_end <= TRAINING_CUTOFF, 'Validation/model-selection split leaks beyond the Dec 2022 cutoff'
assert spec.test_start > TRAINING_CUTOFF, 'Main test split must start after Dec 2022'

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

print(f'Dataset  : {dataset.name}  ({len(TRADABLE_TICKERS)} tradable tickers)')
print(f'Benchmark: {BENCHMARK_TICKER}')
print(f'Train    : {spec.train_start} -> {spec.train_end}')
print(f'Val      : {spec.val_start}   -> {spec.val_end}')
print(f'Test     : {spec.test_start}  -> {spec.test_end}')
print(f'Device   : {DEVICE}')
print(f'Run name : {RUN_NAME}')
print(f'Dynamic N: {DYNAMIC_TOP_N}; cap defensive/base/risk-on = {DEFENSIVE_MAX_WEIGHT:.1%}/{MAX_WEIGHT:.1%}/{RISK_ON_MAX_WEIGHT:.1%}')
print(f'Cutoff   : no training/model selection beyond {TRAINING_CUTOFF}')


## 3. Load Prices

In [ ]:
prices = load_prices(DATASET_NAME, repo_root=repo_root)
print('Shape     :', prices.shape)
print('Date range:', prices['date'].min().date(), '→', prices['date'].max().date())
prices.head(3)

## 4. Features — Shared Toolkit + v4 Ranking/Regime Features

The baseline used 16 shared features plus `price_accel`. v4 keeps those signals and adds features aligned with the task review: cross-sectional ranks/z-scores, benchmark trend and drawdown state, universe breadth and dispersion, volume shock, volatility compression, and relative strength.


In [ ]:
frame = build_model_features(prices)

BASE_FEATURES = [
    'return_1d', 'return_5d', 'return_10d', 'return_20d', 'return_60d',
    'vol_5d', 'vol_10d', 'vol_20d', 'vol_60d',
    'downside_vol_20d', 'upside_vol_20d', 'atr_14',
    'momentum_5d', 'momentum_10d', 'momentum_20d', 'momentum_60d', 'momentum_120d',
    'rsi_14', 'macd_hist', 'bollinger_z_20d',
    'beta_20d_spy', 'beta_60d_spy',
    'excess_return_5d_vs_spy', 'excess_return_20d_vs_spy', 'excess_return_60d_vs_spy',
    'relative_momentum_20d_vs_spy',
    'volume_zscore_20d', 'volume_zscore_60d', 'dollar_volume_ratio_20d',
    'price_to_sma_20d', 'price_to_sma_50d', 'price_to_sma_200d',
    'intraday_range', 'close_open_gap', 'close_location_in_range',
    'distance_to_20d_high', 'distance_to_20d_low', 'distance_to_60d_high', 'distance_to_60d_low',
    'skew_20d', 'kurtosis_20d',
]

CUSTOM_FEATURES = [
    'price_accel', 'momentum_reversal_20_60', 'vol_compression_20_60',
]

REGIME_FEATURES = [
    'spy_return_5d', 'spy_return_20d', 'spy_vol_20d', 'spy_vol_60d',
    'spy_trend_20d', 'spy_trend_60d', 'spy_drawdown_60d', 'spy_rebound_5d',
    'universe_breadth_20d', 'universe_dispersion_5d', 'universe_dispersion_20d',
    'universe_median_return_5d', 'universe_median_return_20d',
    'high_vol_regime', 'drawdown_regime', 'rebound_regime',
]

CS_FEATURES = [
    c for c in frame.columns
    if c.endswith('_cs_rank') or c.endswith('_cs_z')
]

ALL_FEATURES = BASE_FEATURES + CUSTOM_FEATURES + REGIME_FEATURES + CS_FEATURES
missing_features = sorted(set(ALL_FEATURES) - set(frame.columns))
assert not missing_features, f'Missing features: {missing_features}'

print(f'Base features      : {len(BASE_FEATURES)}')
print(f'Custom features    : {len(CUSTOM_FEATURES)}')
print(f'Regime features    : {len(REGIME_FEATURES)}')
print(f'Cross-sec features : {len(CS_FEATURES)}')
print(f'Total features     : {len(ALL_FEATURES)}')

frame[['date', 'ticker', 'price_accel', 'spy_drawdown_60d', 'universe_dispersion_20d']].dropna().head(3)


## 5. Targets + Train / Val / Test Split

v4 keeps the 5-day SPY-relative alpha target, then adds multi-horizon alpha labels, top/bottom quintile labels, realized volatility, downside volatility, and worst-path drawdown. The extra labels give the confidence layer a job: identify upside concentration candidates and reject high-tail-risk losers.


In [ ]:
TARGET_COL = f'forward_alpha_{HORIZON}d_vs_spy'
FWD_RETURN_COL = f'forward_return_{HORIZON}d'
VOL_TARGET_COL = f'forward_realized_vol_{HORIZON}d'
DOWNSIDE_TARGET_COL = f'forward_downside_vol_{HORIZON}d'
WORST_PATH_COL = f'forward_worst_path_{HORIZON}d'
TARGET_RANK_COL = 'target_alpha_cs_rank'
TOP_QUINTILE_COL = 'target_top_quintile'
BOTTOM_QUINTILE_COL = 'target_bottom_quintile'
AUX_TARGET_COLS = [f'forward_alpha_{h}d_vs_spy' for h in AUX_HORIZONS]

alpha_targets = make_forward_alpha_target(prices, horizon=HORIZON)

panel = prices.sort_values(['ticker', 'date']).copy()
alpha_benchmark = ALPHA_BENCHMARK_TICKER if ALPHA_BENCHMARK_TICKER in set(panel['ticker']) else BENCHMARK_TICKER

for horizon in [HORIZON] + AUX_HORIZONS:
    fwd_col = f'forward_return_{horizon}d'
    panel[fwd_col] = panel.groupby('ticker')['adj_close'].shift(-horizon) / panel['adj_close'] - 1.0
    if horizon in AUX_HORIZONS:
        bench = panel.loc[panel['ticker'] == alpha_benchmark, ['date', fwd_col]].rename(columns={fwd_col: 'bench_forward_return'})
        panel = panel.merge(bench, on='date', how='left')
        panel[f'forward_alpha_{horizon}d_vs_spy'] = panel[fwd_col] - panel['bench_forward_return']
        panel = panel.drop(columns=['bench_forward_return'])

future_returns = []
for step in range(1, HORIZON + 1):
    future_returns.append(
        panel.groupby('ticker')['adj_close'].shift(-step) / panel.groupby('ticker')['adj_close'].shift(-(step - 1)) - 1.0
    )
future_return_frame = pd.concat(future_returns, axis=1)
panel[VOL_TARGET_COL] = future_return_frame.std(axis=1, ddof=0) * np.sqrt(252.0)
panel[DOWNSIDE_TARGET_COL] = future_return_frame.clip(upper=0.0).std(axis=1, ddof=0) * np.sqrt(252.0)
panel[WORST_PATH_COL] = (1.0 + future_return_frame).cumprod(axis=1).min(axis=1) - 1.0

risk_cols = ['date', 'ticker', FWD_RETURN_COL, VOL_TARGET_COL, DOWNSIDE_TARGET_COL, WORST_PATH_COL] + AUX_TARGET_COLS
risk_targets = panel[risk_cols]

target_frame = (
    frame
    .merge(alpha_targets, on=['date', 'ticker'], how='left')
    .merge(risk_targets, on=['date', 'ticker'], how='left')
)
target_frame = target_frame[target_frame['ticker'].isin(TRADABLE_TICKERS)].copy()

target_frame[TARGET_RANK_COL] = target_frame.groupby('date')[TARGET_COL].rank(pct=True, method='average')
target_frame[TOP_QUINTILE_COL] = (target_frame[TARGET_RANK_COL] > 0.80).astype(float)
target_frame[BOTTOM_QUINTILE_COL] = (target_frame[TARGET_RANK_COL] < 0.20).astype(float)
target_frame['target_alpha_cs_z'] = (
    target_frame[TARGET_COL] - target_frame.groupby('date')[TARGET_COL].transform('mean')
) / target_frame.groupby('date')[TARGET_COL].transform('std').replace(0.0, np.nan)

target_cols = [
    TARGET_COL, TARGET_RANK_COL, TOP_QUINTILE_COL, BOTTOM_QUINTILE_COL,
    VOL_TARGET_COL, DOWNSIDE_TARGET_COL, WORST_PATH_COL, *AUX_TARGET_COLS,
]
target_frame = target_frame.replace([np.inf, -np.inf], np.nan).dropna(subset=ALL_FEATURES + target_cols).reset_index(drop=True)

train = slice_split(target_frame, DATASET_NAME, 'train', repo_root=repo_root)
val   = slice_split(target_frame, DATASET_NAME, 'val',   repo_root=repo_root)
test  = slice_split(target_frame, DATASET_NAME, 'test',  repo_root=repo_root)

train_mean = train[ALL_FEATURES].mean()
train_std  = train[ALL_FEATURES].std(ddof=0).replace(0.0, 1.0)

def scale(df):
    return ((df[ALL_FEATURES] - train_mean) / train_std).to_numpy(dtype=np.float32)

X_train, X_val, X_test = scale(train), scale(val), scale(test)

print(f'Train : {len(train):>6,} rows    X_train : {X_train.shape}')
print(f'Val   : {len(val):>6,} rows    X_val   : {X_val.shape}')
print(f'Test  : {len(test):>6,} rows    X_test  : {X_test.shape}')
print(f'Top-quintile train rate   : {train[TOP_QUINTILE_COL].mean():.2%}')
print(f'Bottom-quintile train rate: {train[BOTTOM_QUINTILE_COL].mean():.2%}')


## 6. Build Per-Ticker Sequences

Sequences still stay inside ticker boundaries. The label bundle is taken from the last timestep of each sequence and now includes alpha, rank, top/bottom class, volatility, downside risk, worst-path return, and auxiliary 10/20-day alpha.


In [ ]:
def make_sequences(
    df: pd.DataFrame,
    X: np.ndarray,
    target_columns: list[str],
    seq_len: int,
):
    """Return (X_seq, y_dict, meta) where meta is list of (date, ticker)."""
    Xs, ys, meta = [], {col: [] for col in target_columns}, []
    df = df.reset_index(drop=True)
    for ticker, grp in df.groupby('ticker', sort=False):
        idx = grp.index.tolist()
        dates = grp['date'].tolist()
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx)):
            rows = idx[end - seq_len + 1 : end + 1]
            row_idx = idx[end]
            Xs.append(X[rows])
            for col in target_columns:
                ys[col].append(df.loc[row_idx, col])
            meta.append((dates[end], ticker))

    y_dict = {col: np.asarray(vals, dtype=np.float32) for col, vals in ys.items()}
    return np.stack(Xs).astype(np.float32), y_dict, meta

TARGET_COLUMNS = [
    TARGET_COL, TARGET_RANK_COL, TOP_QUINTILE_COL, BOTTOM_QUINTILE_COL,
    VOL_TARGET_COL, DOWNSIDE_TARGET_COL, WORST_PATH_COL, *AUX_TARGET_COLS,
]

print('Building sequences ...')
X_tr_seq, y_tr_seq, meta_train = make_sequences(train, X_train, TARGET_COLUMNS, SEQ_LEN)
X_va_seq, y_va_seq, meta_val   = make_sequences(val,   X_val,   TARGET_COLUMNS, SEQ_LEN)
X_te_seq, y_te_seq, meta_test  = make_sequences(test,  X_test,  TARGET_COLUMNS, SEQ_LEN)

y_train = y_tr_seq[TARGET_COL]
y_val   = y_va_seq[TARGET_COL]

print(f'Train sequences : {X_tr_seq.shape}')
print(f'Val sequences   : {X_va_seq.shape}')
print(f'Test sequences  : {X_te_seq.shape}')
print(f'Train top-quintile sequences   : {y_tr_seq[TOP_QUINTILE_COL].mean():.2%}')
print(f'Train bottom-quintile sequences: {y_tr_seq[BOTTOM_QUINTILE_COL].mean():.2%}')


## 7. Ranking Confidence LSTM

The baseline LSTM predicted one scalar alpha with MSE. v4 keeps that alpha head but adds top/bottom confidence, volatility, downside, worst-path, and auxiliary 10/20-day alpha heads. Training combines pointwise alpha error, pairwise ranking pressure, upside classification, loser classification, and risk-head losses so confidence can become an allocation control.


In [ ]:
class RankingConfidenceLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2, aux_horizons=None):
        super().__init__()
        self.aux_horizons = list(aux_horizons or [])
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.alpha_head = nn.Linear(hidden_size, 1)
        self.confidence_head = nn.Linear(hidden_size, 1)
        self.bottom_head = nn.Linear(hidden_size, 1)
        self.volatility_head = nn.Linear(hidden_size, 1)
        self.downside_head = nn.Linear(hidden_size, 1)
        self.worst_path_head = nn.Linear(hidden_size, 1)
        self.aux_alpha_heads = nn.ModuleDict({str(h): nn.Linear(hidden_size, 1) for h in self.aux_horizons})
        for head in [
            self.alpha_head, self.confidence_head, self.bottom_head,
            self.volatility_head, self.downside_head, self.worst_path_head,
            *self.aux_alpha_heads.values(),
        ]:
            nn.init.xavier_uniform_(head.weight)
            nn.init.zeros_(head.bias)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        h = self.dropout(self.norm(h_n[-1]))
        outputs = {
            'alpha': self.alpha_head(h).squeeze(-1),
            'confidence_logit': self.confidence_head(h).squeeze(-1),
            'bottom_logit': self.bottom_head(h).squeeze(-1),
            'volatility': self.volatility_head(h).squeeze(-1),
            'downside': self.downside_head(h).squeeze(-1),
            'worst_path': self.worst_path_head(h).squeeze(-1),
        }
        for horizon, head in self.aux_alpha_heads.items():
            outputs[f'alpha_{horizon}d'] = head(h).squeeze(-1)
        return outputs


# Backward-compatible alias for checkpoint reproduction cells.
BaselineLSTM = RankingConfidenceLSTM


def _as_tensor_dict(y_dict, start, stop, device):
    return {
        key: torch.as_tensor(val[start:stop], dtype=torch.float32, device=device)
        for key, val in y_dict.items()
    }


def _pairwise_rank_loss(pred, actual, max_pairs=4096):
    n = pred.shape[0]
    if n < 2:
        return pred.new_tensor(0.0)
    pair_count = min(max_pairs, n * 4)
    i = torch.randint(0, n, (pair_count,), device=pred.device)
    j = torch.randint(0, n, (pair_count,), device=pred.device)
    direction = torch.sign(actual[i] - actual[j])
    mask = direction != 0
    if not mask.any():
        return pred.new_tensor(0.0)
    margin = (pred[i] - pred[j]) * direction
    return torch.nn.functional.softplus(-margin[mask]).mean()


def _loss_from_outputs(outputs, yb):
    alpha_loss = torch.nn.functional.mse_loss(outputs['alpha'], yb[TARGET_COL])
    rank_loss = _pairwise_rank_loss(outputs['alpha'], yb[TARGET_COL])
    top_loss = torch.nn.functional.binary_cross_entropy_with_logits(
        outputs['confidence_logit'], yb[TOP_QUINTILE_COL]
    )
    bottom_loss = torch.nn.functional.binary_cross_entropy_with_logits(
        outputs['bottom_logit'], yb[BOTTOM_QUINTILE_COL]
    )
    vol_loss = torch.nn.functional.mse_loss(
        torch.nn.functional.softplus(outputs['volatility']), yb[VOL_TARGET_COL]
    )
    downside_loss = torch.nn.functional.mse_loss(
        torch.nn.functional.softplus(outputs['downside']), yb[DOWNSIDE_TARGET_COL]
    )
    worst_path_loss = torch.nn.functional.mse_loss(
        -torch.nn.functional.softplus(outputs['worst_path']), yb[WORST_PATH_COL]
    )
    aux_losses = []
    for horizon, target_col in zip(AUX_HORIZONS, AUX_TARGET_COLS):
        key = f'alpha_{horizon}d'
        if key in outputs and target_col in yb:
            aux_losses.append(torch.nn.functional.mse_loss(outputs[key], yb[target_col]))
    aux_loss = torch.stack(aux_losses).mean() if aux_losses else alpha_loss.new_tensor(0.0)
    total = (
        ALPHA_LOSS_WEIGHT * alpha_loss
        + RANK_LOSS_WEIGHT * rank_loss
        + TOP_QUINTILE_WEIGHT * top_loss
        + BOTTOM_QUINTILE_WEIGHT * bottom_loss
        + VOL_LOSS_WEIGHT * vol_loss
        + DOWNSIDE_LOSS_WEIGHT * downside_loss
        + WORST_PATH_LOSS_WEIGHT * worst_path_loss
        + AUX_ALPHA_LOSS_WEIGHT * aux_loss
    )
    return total, {
        'alpha_loss': float(alpha_loss.detach().cpu()),
        'rank_loss': float(rank_loss.detach().cpu()),
        'top_loss': float(top_loss.detach().cpu()),
        'bottom_loss': float(bottom_loss.detach().cpu()),
        'vol_loss': float(vol_loss.detach().cpu()),
        'downside_loss': float(downside_loss.detach().cpu()),
        'worst_path_loss': float(worst_path_loss.detach().cpu()),
        'aux_loss': float(aux_loss.detach().cpu()),
    }


def _predict_batched(model, X, device, batch_size=2048):
    model.eval()
    parts = {'alpha': [], 'confidence': [], 'bottom_probability': [], 'volatility': [], 'downside': [], 'worst_path': []}
    for horizon in AUX_HORIZONS:
        parts[f'alpha_{horizon}d'] = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            Xb = torch.as_tensor(X[i:i+batch_size], dtype=torch.float32, device=device)
            out = _model_outputs(model, Xb)
            parts['alpha'].append(out['alpha'].detach().cpu().numpy())
            parts['confidence'].append(torch.sigmoid(out['confidence_logit']).detach().cpu().numpy())
            parts['bottom_probability'].append(torch.sigmoid(out['bottom_logit']).detach().cpu().numpy())
            parts['volatility'].append(torch.nn.functional.softplus(out['volatility']).detach().cpu().numpy())
            parts['downside'].append(torch.nn.functional.softplus(out['downside']).detach().cpu().numpy())
            parts['worst_path'].append((-torch.nn.functional.softplus(out['worst_path'])).detach().cpu().numpy())
            for horizon in AUX_HORIZONS:
                key = f'alpha_{horizon}d'
                if key in out:
                    parts[key].append(out[key].detach().cpu().numpy())
    return {key: np.concatenate(vals) for key, vals in parts.items() if vals}


def _eval_loss_batched(model, X, y_dict, device, batch_size=2048):
    total, n = 0.0, 0
    model.eval()
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            stop = min(i + batch_size, len(X))
            Xb = torch.as_tensor(X[i:stop], dtype=torch.float32, device=device)
            yb = _as_tensor_dict(y_dict, i, stop, device)
            loss, _ = _loss_from_outputs(_model_outputs(model, Xb), yb)
            total += float(loss) * (stop - i)
            n += stop - i
    return total / max(n, 1)


def train_lstm(model, X_tr, y_tr, X_va, y_va, epochs, batch_size, lr, device):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    order = np.arange(len(X_tr))
    history = []

    for epoch in range(epochs):
        model.train()
        np.random.shuffle(order)
        batch_losses = []
        for start in range(0, len(order), batch_size):
            batch_idx = order[start:start + batch_size]
            Xb = torch.as_tensor(X_tr[batch_idx], dtype=torch.float32, device=device)
            yb = {
                key: torch.as_tensor(val[batch_idx], dtype=torch.float32, device=device)
                for key, val in y_tr.items()
            }
            optimizer.zero_grad()
            loss, _ = _loss_from_outputs(_model_outputs(model, Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            batch_losses.append(float(loss.detach().cpu()))

        tr_loss = _eval_loss_batched(model, X_tr, y_tr, device)
        va_loss = _eval_loss_batched(model, X_va, y_va, device)
        history.append({'epoch': epoch + 1, 'train_loss': tr_loss, 'val_loss': va_loss, 'batch_loss': np.mean(batch_losses)})
        if (epoch + 1) % 5 == 0:
            print(f'  epoch {epoch+1:3d}/{epochs}  train={tr_loss:.6f}  val={va_loss:.6f}')

    return pd.DataFrame(history)


print('Device:', DEVICE)
print(f'Input size: {X_tr_seq.shape[2]}')


## 8. Train

In [ ]:
model = RankingConfidenceLSTM(
    input_size   = X_tr_seq.shape[2],
    hidden_size  = HIDDEN_SIZE,
    num_layers   = NUM_LAYERS,
    dropout      = DROPOUT,
    aux_horizons = AUX_HORIZONS,
)

print(f'Training on {X_tr_seq.shape[0]:,} sequences for {EPOCHS} epochs ...')
history = train_lstm(
    model, X_tr_seq, y_tr_seq, X_va_seq, y_va_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE,
)

best_val = history['val_loss'].min()
print(f'\nBest val composite loss: {best_val:.8f}')
history.tail()


In [ ]:
# Validation Diagnostics
from scipy import stats
from sklearn.metrics import roc_auc_score

print('=' * 60)
print('VALIDATION DIAGNOSTICS')
print('=' * 60)

val_outputs = _predict_batched(model, X_va_seq, DEVICE)
val_preds = val_outputs['alpha']
val_confidence = val_outputs['confidence']
val_bottom_probability = val_outputs.get('bottom_probability', np.full_like(val_confidence, 0.2))
val_volatility = val_outputs.get('volatility', np.zeros_like(val_confidence))
val_downside = val_outputs.get('downside', np.zeros_like(val_confidence))
val_worst_path = val_outputs.get('worst_path', np.zeros_like(val_confidence))

naive_zero_mse  = float(np.mean(y_va_seq[TARGET_COL] ** 2))
naive_mean      = float(y_tr_seq[TARGET_COL].mean())
naive_mean_mse  = float(np.mean((y_va_seq[TARGET_COL] - naive_mean) ** 2))
lstm_val_mse    = float(np.mean((y_va_seq[TARGET_COL] - val_preds) ** 2))

print('\n-- 1. Pointwise Baseline Comparison --')
print(f'  Naive zero-predict MSE  : {naive_zero_mse:.8f}')
print(f'  Naive mean-predict MSE  : {naive_mean_mse:.8f}')
print(f'  LSTM alpha MSE          : {lstm_val_mse:.8f}')

print('\n-- 2. Directional Accuracy --')
actual_direction = np.sign(y_va_seq[TARGET_COL])
predicted_direction = np.sign(val_preds)
nonzero_mask = actual_direction != 0
dir_accuracy = float((actual_direction[nonzero_mask] == predicted_direction[nonzero_mask]).mean())
print(f'  Directional accuracy : {dir_accuracy:.4f}  ({dir_accuracy*100:.1f}%)')

print('\n-- 3. Ranking Quality --')
ic, ic_pvalue = stats.spearmanr(val_preds, y_va_seq[TARGET_COL])
ic = float(ic)
print(f'  Overall IC      : {ic:.4f}')
print(f'  IC p-value      : {ic_pvalue:.4f}')
try:
    top_auc = float(roc_auc_score(y_va_seq[TOP_QUINTILE_COL], val_confidence))
except ValueError:
    top_auc = np.nan
try:
    bottom_auc = float(roc_auc_score(y_va_seq[BOTTOM_QUINTILE_COL], val_bottom_probability))
except ValueError:
    bottom_auc = np.nan
print(f'  Top-quintile AUC   : {top_auc:.4f}')
print(f'  Bottom-quintile AUC: {bottom_auc:.4f}')

val_dates = pd.to_datetime([m[0] for m in meta_val])
val_diag = pd.DataFrame({
    'date': val_dates,
    'predicted': val_preds,
    'confidence': val_confidence,
    'bottom_probability': val_bottom_probability,
    'predicted_volatility': val_volatility,
    'predicted_downside': val_downside,
    'predicted_worst_path': val_worst_path,
    'actual': y_va_seq[TARGET_COL],
    'top_quintile': y_va_seq[TOP_QUINTILE_COL],
    'bottom_quintile': y_va_seq[BOTTOM_QUINTILE_COL],
    'actual_volatility': y_va_seq[VOL_TARGET_COL],
    'actual_downside': y_va_seq[DOWNSIDE_TARGET_COL],
    'actual_worst_path': y_va_seq[WORST_PATH_COL],
})
val_diag['abs_error'] = (val_diag['actual'] - val_diag['predicted']).abs()

print('\n-- 4. IC Stability by Month --')
ic_by_month = (
    val_diag.assign(month=lambda df: df['date'].dt.to_period('M'))
    .groupby('month')
    .apply(lambda g: float(stats.spearmanr(g['predicted'], g['actual'])[0]))
    .rename('monthly_ic')
)
positive_months = int((ic_by_month > 0).sum())
total_months = int(len(ic_by_month))
ic_hit_rate = positive_months / total_months
print(f'  Mean monthly IC   : {ic_by_month.mean():.4f}')
print(f'  Std monthly IC    : {ic_by_month.std():.4f}')
print(f'  IC hit rate       : {positive_months}/{total_months} positive ({ic_hit_rate:.1%})')
print(ic_by_month.to_string())

print('\n-- 5. Regime Diagnostics --')
regime_lookup = target_frame[['date', 'high_vol_regime', 'drawdown_regime', 'rebound_regime']].drop_duplicates('date')
regime_df = val_diag.merge(regime_lookup, on='date', how='left')
for regime_col in ['high_vol_regime', 'drawdown_regime', 'rebound_regime']:
    grp = regime_df.loc[regime_df[regime_col] == 1.0]
    if len(grp) < 20:
        continue
    regime_ic, _ = stats.spearmanr(grp['predicted'], grp['actual'])
    print(f'  {regime_col:<16s}: IC = {float(regime_ic): .4f}  n={len(grp):,}')

print('\n-- 6. Confidence Calibration Deciles --')
val_diag['confidence_decile'] = pd.qcut(val_diag['confidence'], q=10, labels=False, duplicates='drop')
conf_summary = val_diag.groupby('confidence_decile').agg(
    avg_confidence=('confidence', 'mean'),
    avg_actual_alpha=('actual', 'mean'),
    hit_rate=('actual', lambda s: float((s > 0).mean())),
    top_quintile_rate=('top_quintile', 'mean'),
    avg_abs_error=('abs_error', 'mean'),
    avg_actual_downside=('actual_downside', 'mean'),
    avg_actual_worst_path=('actual_worst_path', 'mean'),
    n=('actual', 'size'),
)
print(conf_summary.to_string())

print('\n-- 7. Loser / Risk Calibration --')
val_diag['bottom_decile'] = pd.qcut(val_diag['bottom_probability'], q=10, labels=False, duplicates='drop')
risk_summary = val_diag.groupby('bottom_decile').agg(
    avg_bottom_probability=('bottom_probability', 'mean'),
    avg_actual_alpha=('actual', 'mean'),
    bottom_quintile_rate=('bottom_quintile', 'mean'),
    avg_actual_downside=('actual_downside', 'mean'),
    avg_actual_worst_path=('actual_worst_path', 'mean'),
    n=('actual', 'size'),
)
print(risk_summary.to_string())

_diagnostic_metrics = {
    'val_alpha_mse': lstm_val_mse,
    'val_directional_accuracy': dir_accuracy,
    'val_ic': ic,
    'val_ic_hit_rate_monthly': ic_hit_rate,
    'val_top_quintile_auc': top_auc if np.isfinite(top_auc) else 0.0,
    'val_bottom_quintile_auc': bottom_auc if np.isfinite(bottom_auc) else 0.0,
    'val_confidence_top_decile_alpha': float(conf_summary['avg_actual_alpha'].iloc[-1]),
    'val_confidence_bottom_decile_alpha': float(conf_summary['avg_actual_alpha'].iloc[0]),
    'val_bottom_prob_top_decile_alpha': float(risk_summary['avg_actual_alpha'].iloc[-1]),
}

print('\n' + '=' * 60)
print('DIAGNOSTIC SUMMARY')
print('=' * 60)
print(f'  Alpha MSE              : {lstm_val_mse:.8f}')
print(f'  Directional accuracy   : {dir_accuracy*100:.1f}%')
print(f'  IC overall             : {ic:.4f}')
print(f'  IC hit rate monthly    : {ic_hit_rate:.1%}')
print(f'  Top-quintile AUC       : {top_auc:.4f}')
print(f'  Bottom-quintile AUC    : {bottom_auc:.4f}')
print('=' * 60)


---------------------------------------------------------------------------------

### Checkpoint saving
Save Model Weights + `log_model_submission`

In [ ]:
# ── Save model checkpoint + metadata ──────────────────────────────────────────
import json
import torch

checkpoint_dir = artifact_dir / 'model'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Single bundled checkpoint — contains everything needed to reload the model
checkpoint_path = checkpoint_dir / 'lstm_checkpoint.pt'
torch.save(
    {
        'state_dict'          : model.state_dict(),
        'feature_names'       : ALL_FEATURES,
        'horizon'             : HORIZON,
        'aux_horizons'        : AUX_HORIZONS,
        'rebalance_frequency' : 'every_5_trading_days',
        'model_config': {
            'architecture' : 'RankingConfidenceLSTM',
            'input_dim'    : len(ALL_FEATURES),
            'hidden_dim'   : HIDDEN_SIZE,
            'num_layers'   : NUM_LAYERS,
            'dropout'      : DROPOUT,
            'aux_horizons' : AUX_HORIZONS,
        },
        'preprocessing': {
            'scaler' : 'train_mean_std',
            'means'  : train_mean.to_dict(),
            'stds'   : train_std.to_dict(),
        },
        'training': {
            'epochs'       : EPOCHS,
            'batch_size'   : BATCH_SIZE,
            'lr'           : LR,
            'seed'         : SEED,
            'best_val_mse' : round(float(best_val), 8),
        },
        'data': {
            'dataset_name'   : dataset.name if hasattr(dataset, 'name') else str(dataset),
            'tickers'        : list(spec.tickers),
            'benchmark'      : str(spec.benchmark_ticker),
            'alpha_benchmark': ALPHA_BENCHMARK_TICKER,
            'train_end'      : str(spec.train_end),
            'val_end'        : str(spec.val_end),
            'test_start'     : str(spec.test_start),
            'target'         : TARGET_COL,
            'aux_targets'    : AUX_TARGET_COLS,
            'seq_len'        : SEQ_LEN,
            'custom_features': CUSTOM_FEATURES,
            'regime_features': REGIME_FEATURES,
            'cross_sectional_feature_count': len(CS_FEATURES),
        },
        'source_notebook' : 'baseline_lstm.ipynb',
        'run_name'        : RUN_NAME,
        'author'          : 'Minho',
        'loss_weights': {
            'alpha': ALPHA_LOSS_WEIGHT,
            'rank': RANK_LOSS_WEIGHT,
            'top_quintile': TOP_QUINTILE_WEIGHT,
            'bottom_quintile': BOTTOM_QUINTILE_WEIGHT,
            'volatility': VOL_LOSS_WEIGHT,
            'downside': DOWNSIDE_LOSS_WEIGHT,
            'worst_path': WORST_PATH_LOSS_WEIGHT,
            'aux_alpha': AUX_ALPHA_LOSS_WEIGHT,
        },
        'portfolio_config': {
            'dynamic_top_n': DYNAMIC_TOP_N,
            'sleeve_sizes': SLEEVE_SIZES,
            'confidence_threshold': CONFIDENCE_THRESHOLD,
            'high_confidence_threshold': HIGH_CONFIDENCE_THRESHOLD,
            'low_confidence_threshold': LOW_CONFIDENCE_THRESHOLD,
            'spread_threshold': SPREAD_THRESHOLD,
            'risk_quality_threshold': RISK_QUALITY_THRESHOLD,
            'max_weight': MAX_WEIGHT,
            'defensive_max_weight': DEFENSIVE_MAX_WEIGHT,
            'risk_on_max_weight': RISK_ON_MAX_WEIGHT,
            'min_active_names': MIN_ACTIVE_NAMES,
            'loser_rank_cutoff': LOSER_RANK_CUTOFF,
            'bottom_prob_cutoff': BOTTOM_PROB_CUTOFF,
            'no_trade_band': NO_TRADE_BAND,
        },
    },
    checkpoint_path,
)
print(f'Bundled checkpoint saved → {checkpoint_path}')

# Reconstruction recipe
recipe = """
# How to reproduce Minho v4 RankingConfidence LSTM predictions from scratch
#
# Requirements: repo installed, venv312 active, MLflow artifacts downloaded
#
# Steps:
#   1. Load the bundled checkpoint
#      bundle = torch.load('lstm_checkpoint.pt', map_location='cpu')
#
#   2. Rebuild model architecture
#      model = BaselineLSTM(
#          input_size   = bundle['model_config']['input_dim'],
#          hidden_size  = bundle['model_config']['hidden_dim'],
#          num_layers   = bundle['model_config']['num_layers'],
#          dropout      = bundle['model_config']['dropout'],
#          aux_horizons = bundle['model_config'].get('aux_horizons', []),
#      )
#      model.load_state_dict(bundle['state_dict'])
#      model.eval()
#
#   3. Load preprocessing
#      train_mean = pd.Series(bundle['preprocessing']['means'])
#      train_std  = pd.Series(bundle['preprocessing']['stds'])
#
#   4. Load prices and call predict_from_prices()
#      prices = load_prices(bundle['data']['dataset_name'])
#      predictions = predict_from_prices(
#          model=model, prices=prices,
#          train_mean=train_mean, train_std=train_std,
#          feature_names=bundle['feature_names'],
#          seq_len=bundle['data']['seq_len'],
#          horizon=bundle['horizon'],
#      )
"""
recipe_path = checkpoint_dir / 'REPRODUCE.py'
recipe_path.write_text(recipe)
print(f'Reproduction recipe saved → {recipe_path}')

print('\nAll model artifacts saved:')
for p in sorted(checkpoint_dir.iterdir()):
    print(f'  {p.name}')


---------------------------------------------------------------------------------

## 9. Predictions

In [ ]:
# Generate predictions — SPY explicitly excluded via TRADABLE_TICKERS
predictions = predict_from_prices(
    model         = model,
    prices        = prices,
    train_mean    = train_mean,
    train_std     = train_std,
    feature_names = ALL_FEATURES,
    seq_len       = SEQ_LEN,
    horizon       = HORIZON,
    tickers       = TRADABLE_TICKERS,
)

# Filter to test split dates only
test_dates = set(test['date'].unique())
predictions = predictions[
    predictions['date'].isin(test_dates)
].reset_index(drop=True)

# Filter to rebalance dates only — must match portfolio weight dates
# so MLflow predictions and weights are consistent
all_pred_dates = sorted(predictions['date'].unique())
rebal_dates    = set(pd.DatetimeIndex(all_pred_dates)[::REBALANCE_FREQ])
predictions    = predictions[
    predictions['date'].isin(rebal_dates)
].reset_index(drop=True)

# Validate
assert BENCHMARK_TICKER not in predictions['ticker'].values, \
    f'{BENCHMARK_TICKER} found in predictions — exclusion failed'

print('Predictions shape :', predictions.shape)
print('Rebalance dates   :', len(rebal_dates))
print('SPY in predictions:', BENCHMARK_TICKER in predictions['ticker'].values)
predictions.head()

## 10. Confidence-Aware Portfolio Weights

v4 turns confidence into active risk budget. The allocator first removes obvious loser/tail-risk candidates, then chooses high/normal/low concentration from confidence, predicted spread, and risk quality. High-confidence books can hold fewer names with a looser cap; low-confidence books become broader, cooler softmax, and more tightly capped.


In [ ]:
# Confidence-aware portfolio construction
from portfolio_toolkit.contracts import PortfolioWeights as _PW


def _effective_names(weights: pd.Series) -> float:
    w = weights[weights > 0].astype(float)
    if w.empty:
        return 0.0
    return float(1.0 / np.square(w).sum())


def _project_capped_simplex(weights: pd.Series, max_weight: float) -> pd.Series:
    """Project long-only weights onto sum=1 with a hard per-name cap."""
    w = weights.clip(lower=0.0).astype(float).copy()
    if max_weight <= 0:
        raise ValueError('max_weight must be positive')
    if max_weight * len(w) < 1.0 - 1e-12:
        raise ValueError('max_weight is too small for the number of active names')
    if w.sum() <= 0:
        w[:] = 1.0

    result = pd.Series(0.0, index=w.index, dtype=float)
    remaining = w.index.copy()
    residual = 1.0

    for _ in range(len(w) + 1):
        if len(remaining) == 0 or residual <= 1e-15:
            break
        base = w.loc[remaining]
        if base.sum() <= 0:
            alloc = pd.Series(residual / len(remaining), index=remaining)
        else:
            alloc = base / base.sum() * residual
        over = alloc > max_weight
        if not over.any():
            result.loc[remaining] = alloc
            residual = 0.0
            break
        capped_names = alloc.index[over]
        result.loc[capped_names] = max_weight
        residual -= max_weight * len(capped_names)
        remaining = alloc.index[~over]

    slack = 1.0 - result.sum()
    if abs(slack) > 1e-10:
        capacity = (max_weight - result).clip(lower=0.0)
        if capacity.sum() > 0:
            result += capacity / capacity.sum() * slack
    result = result.clip(lower=0.0, upper=max_weight)
    if abs(result.sum() - 1.0) > 1e-8:
        under = result < max_weight - 1e-12
        if under.any():
            result.loc[under] += (1.0 - result.sum()) * (result.loc[under] / result.loc[under].sum())
    return result.fillna(0.0)


def _softmax_weights(scores: pd.Series, temperature: float) -> pd.Series:
    values = scores.astype(float).replace([np.inf, -np.inf], np.nan).fillna(scores.median())
    values = values - values.max()
    temp = max(float(temperature), 1e-6)
    exp_values = np.exp(values / temp)
    return pd.Series(exp_values, index=scores.index) / exp_values.sum()


def _apply_no_trade_band(current: pd.Series, previous: pd.Series | None, band: float) -> pd.Series:
    if previous is None or band <= 0:
        return current
    previous = previous.reindex(current.index).fillna(0.0)
    adjusted = current.copy()
    keep = (current - previous).abs() < band
    adjusted[keep] = previous[keep]
    if adjusted.sum() <= 0:
        return current
    return adjusted / adjusted.sum()


def _rank01(s: pd.Series, ascending: bool = True) -> pd.Series:
    return s.astype(float).rank(pct=True, method='average', ascending=ascending).fillna(0.5)


def build_confidence_portfolio(
    predictions: pd.DataFrame,
    *,
    strategy_name: str,
    mode: str = 'dynamic',
    top_k: int | None = None,
    dynamic_top_n: dict = DYNAMIC_TOP_N,
    confidence_threshold: float = CONFIDENCE_THRESHOLD,
    high_confidence_threshold: float = HIGH_CONFIDENCE_THRESHOLD,
    low_confidence_threshold: float = LOW_CONFIDENCE_THRESHOLD,
    spread_threshold: float = SPREAD_THRESHOLD,
    risk_quality_threshold: float = RISK_QUALITY_THRESHOLD,
    max_weight: float = MAX_WEIGHT,
    defensive_max_weight: float = DEFENSIVE_MAX_WEIGHT,
    risk_on_max_weight: float = RISK_ON_MAX_WEIGHT,
    min_active_names: int = MIN_ACTIVE_NAMES,
    no_trade_band: float = NO_TRADE_BAND,
    softmax_temperature: float = SOFTMAX_TEMPERATURE,
    defensive_temperature: float = DEFENSIVE_TEMPERATURE,
    risk_aversion: float = RISK_AVERSION,
    downside_aversion: float = DOWNSIDE_AVERSION,
    tail_risk_aversion: float = TAIL_RISK_AVERSION,
    loser_rank_cutoff: float = LOSER_RANK_CUTOFF,
    bottom_prob_cutoff: float = BOTTOM_PROB_CUTOFF,
) -> tuple[_PW, pd.DataFrame]:
    validated = validate_prediction_frame(predictions)
    tickers = sorted(validated['ticker'].unique().tolist())
    weights = pd.DataFrame(0.0, index=pd.DatetimeIndex(sorted(validated['date'].unique())), columns=tickers)
    weights.index.name = 'date'
    diagnostics = []
    prev = None

    for date_value, frame_date in validated.groupby('date', sort=True):
        frame_date = frame_date.copy()
        n = len(frame_date)
        frame_date['confidence'] = frame_date.get('confidence', 0.5)
        frame_date['confidence'] = frame_date['confidence'].fillna(0.5).clip(0.0, 1.0)
        frame_date['bottom_probability'] = frame_date.get('bottom_probability', 0.2)
        frame_date['bottom_probability'] = frame_date['bottom_probability'].fillna(0.2).clip(0.0, 1.0)
        frame_date['expected_volatility'] = frame_date.get('expected_volatility', 0.0)
        frame_date['expected_downside'] = frame_date.get('expected_downside', 0.0)
        frame_date['expected_worst_path'] = frame_date.get('expected_worst_path', 0.0)
        frame_date[['expected_volatility', 'expected_downside', 'expected_worst_path']] = frame_date[[
            'expected_volatility', 'expected_downside', 'expected_worst_path'
        ]].fillna(0.0)

        frame_date['alpha_rank'] = _rank01(frame_date['expected_return'], ascending=True)
        aux_cols = [c for c in frame_date.columns if c.startswith('expected_alpha_') and c.endswith('d')]
        if aux_cols:
            aux_rank = pd.concat([_rank01(frame_date[c], ascending=True) for c in aux_cols], axis=1).mean(axis=1)
        else:
            aux_rank = frame_date['alpha_rank']
        frame_date['multi_horizon_rank'] = 0.70 * frame_date['alpha_rank'] + 0.30 * aux_rank

        tail_risk = (-frame_date['expected_worst_path']).clip(lower=0.0)
        frame_date['raw_risk_load'] = (
            risk_aversion * frame_date['expected_volatility'].clip(lower=0.0)
            + downside_aversion * frame_date['expected_downside'].clip(lower=0.0)
            + tail_risk_aversion * tail_risk
        )
        frame_date['risk_quality'] = _rank01(-frame_date['raw_risk_load'], ascending=True)
        frame_date['risk_penalty'] = np.exp(-frame_date['raw_risk_load'].clip(lower=0.0))
        frame_date['portfolio_score'] = (
            frame_date['multi_horizon_rank']
            * (0.35 + 0.85 * frame_date['confidence'])
            * frame_date['risk_penalty']
            - 0.35 * frame_date['bottom_probability']
        )

        spread = float(frame_date['expected_return'].quantile(0.90) - frame_date['expected_return'].quantile(0.10))
        top_pool_n = max(1, int(np.ceil(n * 0.20)))
        top_pool = frame_date.nlargest(top_pool_n, 'expected_return')
        top_conf = float(top_pool['confidence'].mean())
        top_risk_quality = float(top_pool['risk_quality'].mean())
        risk_on = (
            top_conf >= high_confidence_threshold
            and spread >= spread_threshold
            and top_risk_quality >= risk_quality_threshold
        )
        uncertain = top_conf < low_confidence_threshold or spread < spread_threshold * 0.50

        tradable_pool = frame_date[
            (frame_date['alpha_rank'] > loser_rank_cutoff)
            & (frame_date['bottom_probability'] < bottom_prob_cutoff)
        ].copy()
        if len(tradable_pool) < min_active_names:
            tradable_pool = frame_date.nlargest(min(n, max(min_active_names, dynamic_top_n.get('normal', min_active_names))), 'portfolio_score').copy()

        if mode == 'top_k':
            regime = f'top_{top_k}'
            k = min(len(tradable_pool), max(1, int(top_k or len(tradable_pool))))
            row_cap = max(max_weight, 1.0 / k + 1e-12)
            temperature = softmax_temperature
            concentration_on = True
        elif mode == 'broad':
            regime = 'low'
            k = min(len(tradable_pool), max(min_active_names, dynamic_top_n.get('low', len(tradable_pool))))
            row_cap = max(defensive_max_weight, 1.0 / k + 1e-12)
            temperature = defensive_temperature
            concentration_on = False
        else:
            if risk_on:
                regime = 'high'
                k = min(len(tradable_pool), max(min_active_names, dynamic_top_n['high']))
                row_cap = max(risk_on_max_weight, 1.0 / k + 1e-12)
                temperature = softmax_temperature
                concentration_on = True
            elif uncertain:
                regime = 'low'
                k = min(len(tradable_pool), max(min_active_names, dynamic_top_n['low']))
                row_cap = max(defensive_max_weight, 1.0 / k + 1e-12)
                temperature = defensive_temperature
                concentration_on = False
            else:
                regime = 'normal'
                k = min(len(tradable_pool), max(min_active_names, dynamic_top_n['normal']))
                row_cap = max(max_weight, 1.0 / k + 1e-12)
                temperature = softmax_temperature
                concentration_on = top_conf >= confidence_threshold

        chosen = tradable_pool.nlargest(k, 'portfolio_score').copy()
        raw = _softmax_weights(chosen.set_index('ticker')['portfolio_score'], temperature)
        row = pd.Series(0.0, index=tickers)
        row.loc[raw.index] = _project_capped_simplex(raw, row_cap)
        row = _apply_no_trade_band(row, prev, no_trade_band)
        row = _project_capped_simplex(row, row_cap)
        weights.loc[pd.Timestamp(date_value), row.index] = row.values
        prev = row

        diagnostics.append({
            'date': pd.Timestamp(date_value),
            'mode': mode,
            'regime': regime,
            'candidate_names_after_filter': int(len(tradable_pool)),
            'selected_names': int((row > 0).sum()),
            'effective_names': _effective_names(row),
            'max_weight': float(row.max()),
            'effective_max_weight_cap': float(row_cap),
            'top_confidence': top_conf,
            'top_risk_quality': top_risk_quality,
            'predicted_spread_90_10': spread,
            'concentration_on': bool(concentration_on),
            'risk_on_gate': bool(risk_on),
            'uncertain_gate': bool(uncertain),
            'avg_bottom_probability_selected': float(chosen['bottom_probability'].mean()),
            'avg_risk_quality_selected': float(chosen['risk_quality'].mean()),
        })

    clean_weights = validate_weights_frame(weights)
    return _PW(
        weights=clean_weights,
        dataset_name=DATASET_NAME.name if hasattr(DATASET_NAME, 'name') else DATASET_NAME,
        strategy_name=strategy_name,
        metadata={
            'mode': mode,
            'dynamic_top_n': dynamic_top_n,
            'max_weight': max_weight,
            'defensive_max_weight': defensive_max_weight,
            'risk_on_max_weight': risk_on_max_weight,
            'min_active_names': min_active_names,
            'no_trade_band': no_trade_band,
            'confidence_threshold': confidence_threshold,
            'high_confidence_threshold': high_confidence_threshold,
            'low_confidence_threshold': low_confidence_threshold,
            'spread_threshold': spread_threshold,
            'risk_quality_threshold': risk_quality_threshold,
            'risk_aversion': risk_aversion,
            'downside_aversion': downside_aversion,
            'tail_risk_aversion': tail_risk_aversion,
            'loser_rank_cutoff': loser_rank_cutoff,
            'bottom_prob_cutoff': bottom_prob_cutoff,
        },
    ), pd.DataFrame(diagnostics)


portfolio_candidates = {}
portfolio_diagnostics = {}

portfolio_candidates['broad'], portfolio_diagnostics['broad'] = build_confidence_portfolio(
    predictions, strategy_name=f'{RUN_NAME}_broad', mode='broad'
)
for k in SLEEVE_SIZES:
    portfolio_candidates[f'top_{k}'], portfolio_diagnostics[f'top_{k}'] = build_confidence_portfolio(
        predictions, strategy_name=f'{RUN_NAME}_top_{k}', mode='top_k', top_k=k,
        min_active_names=min(k, len(TRADABLE_TICKERS)), no_trade_band=0.0,
    )
portfolio_candidates['dynamic'], portfolio_diagnostics['dynamic'] = build_confidence_portfolio(
    predictions, strategy_name=RUN_NAME, mode='dynamic'
)

clean_portfolio = portfolio_candidates['dynamic']
portfolio = clean_portfolio
validated_weights = validate_weights_frame(clean_portfolio.weights)
selected_diag = portfolio_diagnostics['dynamic']

print('Selected portfolio : dynamic confidence/risk budget sleeve')
print('Weights shape      :', validated_weights.shape)
print('Avg active names   :', selected_diag['selected_names'].mean())
print('Avg effective names:', selected_diag['effective_names'].mean())
print('Avg max weight     :', selected_diag['max_weight'].mean())
print('Regime mix         :')
print(selected_diag['regime'].value_counts(normalize=True).to_string())
print('Risk-on days       :', selected_diag['risk_on_gate'].mean())
print('Row sums OK        :', np.allclose(validated_weights.sum(axis=1), 1.0))
selected_diag.tail()


----------------------------------------------------------------------------------------

## 11. Backtest

In [ ]:
# ── Backtest ──────────────────────────────────────────────────────────
import bt
from portfolio_toolkit.backtest import (
    _pivot_prices,
    _align_weights_to_prices,
    _mask_unavailable_weights,
    _make_bt_strategy,
    _commission_fn,
    _compute_turnover,
)
from portfolio_toolkit.reporting import build_metrics
from portfolio_toolkit.contracts import BacktestResult

# ── Step 1: Strict price cleaning ─────────────────────────────────────────────
price_wide_raw    = _pivot_prices(prices)
price_wide_filled = price_wide_raw.ffill(limit=5).bfill(limit=5)

clean_mask  = price_wide_filled.notna().all(axis=0)
price_wide  = price_wide_filled.loc[:, clean_mask]

dropped = sorted(set(price_wide_raw.columns) - set(price_wide.columns))
print(f'Tickers dropped (missing prices) : {len(dropped)}')
if dropped:
    print(f'  {dropped}')

available_tickers = [t for t in TRADABLE_TICKERS if t in price_wide.columns]
print(f'Clean tickers for backtest       : {len(available_tickers)}')

assert price_wide[available_tickers].isna().sum().sum() == 0, \
    'NaNs still present — bt will crash'
print('Price matrix is clean — no NaNs detected.')

# ── Step 2: Align and mask strategy weights ────────────────────────────────────
raw_weights     = validate_weights_frame(clean_portfolio.weights)
aligned_weights = _align_weights_to_prices(
    raw_weights.loc[:, [t for t in available_tickers if t in raw_weights.columns]],
    price_wide.index,
)
aligned_weights = _mask_unavailable_weights(
    aligned_weights, price_wide.loc[:, available_tickers]
)

# ── Step 3: Equal weight baseline (no SPY) ────────────────────────────────────
eq_value      = 1.0 / len(available_tickers)
equal_weights = pd.DataFrame(
    eq_value,
    index   = aligned_weights.index,
    columns = available_tickers,
    dtype   = float,
)
equal_weights.index.name = 'date'
equal_weights = _align_weights_to_prices(equal_weights, price_wide.index)
equal_weights = _mask_unavailable_weights(
    equal_weights, price_wide.loc[:, available_tickers]
)

# ── Step 4: SPY benchmark ──────────────────────────────────────────────────────
spy_bench = pd.DataFrame(
    {'SPY': 1.0},
    index = aligned_weights.index,
)
spy_bench.index.name = 'date'
spy_bench = _align_weights_to_prices(spy_bench, price_wide.index)

cost_fn = _commission_fn(spec.cost_bps)

# ── Step 5: Run bt backtests ───────────────────────────────────────────────────
print('\nRunning backtests ...')
backtests = [
    bt.Backtest(
        _make_bt_strategy(RUN_NAME, aligned_weights),
        price_wide[available_tickers],
        commissions=cost_fn,
        integer_positions=False,
    ),
    bt.Backtest(
        _make_bt_strategy('SPY', spy_bench),
        price_wide[['SPY']],
        commissions=cost_fn,
        integer_positions=False,
    ),
    bt.Backtest(
        _make_bt_strategy('equal_weight', equal_weights),
        price_wide[available_tickers],
        commissions=cost_fn,
        integer_positions=False,
    ),
]

bt_result = bt.run(*backtests)

# ── Step 6: Assemble BacktestResult ───────────────────────────────────────────
nav      = bt_result.prices[RUN_NAME].rename('nav')
returns  = nav.pct_change().fillna(0.0).rename('returns')
turnover = _compute_turnover(aligned_weights)

benchmark_returns = pd.DataFrame(index=nav.index)
for col in bt_result.prices.columns:
    if col == RUN_NAME:
        continue
    benchmark_returns[col] = bt_result.prices[col].pct_change().fillna(0.0)

result = BacktestResult(
    strategy_name    = RUN_NAME,
    dataset_name     = dataset.name if hasattr(dataset, 'name') else str(dataset),
    weights          = aligned_weights,
    nav              = nav,
    returns          = returns,
    turnover         = turnover,
    benchmark_returns= benchmark_returns,
    metrics          = {},
)
result.metrics = build_metrics(result)

print('Backtest complete.')
print()
for k, v in sorted(result.metrics.items()):
    print(f'  {k:<35s}: {v:.6f}')

------------------------------------------------------------------------------------------

In [ ]:
# Benchmark and sleeve comparison table
from math import sqrt
from IPython.display import display

print('=' * 80)
print('BENCHMARK + SLEEVE COMPARISON')
print('=' * 80)


def compute_metrics(nav: pd.Series, returns: pd.Series) -> dict:
    total_return  = float(nav.iloc[-1] / nav.iloc[0] - 1.0)
    n_days        = max(len(returns), 1)
    annual_return = float((1.0 + total_return) ** (252.0 / n_days) - 1.0)
    annual_vol    = float(returns.std(ddof=0) * sqrt(252))
    sharpe        = annual_return / annual_vol if annual_vol > 0 else 0.0
    downside      = returns[returns < 0]
    sortino_vol   = float(downside.std(ddof=0) * sqrt(252)) if len(downside) else 0.0
    sortino       = annual_return / sortino_vol if sortino_vol > 0 else 0.0
    drawdown      = nav / nav.cummax() - 1.0
    max_dd        = float(drawdown.min())
    calmar        = annual_return / abs(max_dd) if max_dd < 0 else 0.0
    return {
        'Total Return' : total_return,
        'Annual Return': annual_return,
        'Annual Vol'   : annual_vol,
        'Sharpe'       : sharpe,
        'Sortino'      : sortino,
        'Max Drawdown' : max_dd,
        'Calmar'       : calmar,
    }


# Evaluate sleeves on the same cleaned price matrix used by the selected backtest.
sleeve_backtests = []
sleeve_weight_cache = {}
for name, candidate in portfolio_candidates.items():
    candidate_weights = validate_weights_frame(candidate.weights)
    candidate_aligned = _align_weights_to_prices(
        candidate_weights.loc[:, [t for t in available_tickers if t in candidate_weights.columns]],
        price_wide.index,
    )
    candidate_aligned = _mask_unavailable_weights(
        candidate_aligned, price_wide.loc[:, available_tickers]
    )
    sleeve_weight_cache[name] = candidate_aligned
    sleeve_backtests.append(
        bt.Backtest(
            _make_bt_strategy(name, candidate_aligned),
            price_wide[available_tickers],
            commissions=cost_fn,
            integer_positions=False,
        )
    )

sleeve_bt_result = bt.run(*sleeve_backtests)
sleeve_rows = []
for name in portfolio_candidates:
    sleeve_nav = sleeve_bt_result.prices[name].rename('nav')
    sleeve_returns = sleeve_nav.pct_change().fillna(0.0)
    m = compute_metrics(sleeve_nav / sleeve_nav.iloc[0], sleeve_returns)
    diag = portfolio_diagnostics[name]
    sleeve_rows.append({
        'portfolio': name,
        'total_return': m['Total Return'],
        'annual_return': m['Annual Return'],
        'sharpe': m['Sharpe'],
        'sortino': m['Sortino'],
        'max_drawdown': m['Max Drawdown'],
        'avg_active_names': diag['selected_names'].mean(),
        'avg_effective_names': diag['effective_names'].mean(),
        'avg_max_weight': diag['max_weight'].mean(),
        'risk_on_day_rate': diag['risk_on_gate'].mean() if 'risk_on_gate' in diag else 0.0,
        'low_conf_day_rate': diag['uncertain_gate'].mean() if 'uncertain_gate' in diag else 0.0,
    })

sleeve_summary = pd.DataFrame(sleeve_rows).sort_values('sharpe', ascending=False)
print('\nSleeve comparison')
display(sleeve_summary.style.format({
    'total_return': '{:.1%}',
    'annual_return': '{:.1%}',
    'sharpe': '{:.3f}',
    'sortino': '{:.3f}',
    'max_drawdown': '{:.1%}',
    'avg_active_names': '{:.1f}',
    'avg_effective_names': '{:.1f}',
    'avg_max_weight': '{:.2%}',
    'risk_on_day_rate': '{:.1%}',
    'low_conf_day_rate': '{:.1%}',
}))

# Existing dynamic-vs-benchmark comparison.
spy_returns = result.benchmark_returns['SPY'] if 'SPY' in result.benchmark_returns.columns else result.benchmark_returns.iloc[:, 0]
eq_returns = result.benchmark_returns['equal_weight'] if 'equal_weight' in result.benchmark_returns.columns else None

spy_nav = (1.0 + spy_returns).cumprod()
spy_nav = spy_nav / spy_nav.iloc[0]
if eq_returns is not None:
    eq_nav = (1.0 + eq_returns).cumprod()
    eq_nav = eq_nav / eq_nav.iloc[0]
else:
    eq_returns = spy_returns
    eq_nav = spy_nav
    print('Warning: equal_weight not found in benchmark_returns, using SPY as proxy')

lstm_metrics = compute_metrics(result.nav / result.nav.iloc[0], result.returns)
spy_metrics  = compute_metrics(spy_nav, spy_returns)
eq_metrics   = compute_metrics(eq_nav, eq_returns)

def excess(a, b): return a - b
lstm_vs_spy = {k: excess(lstm_metrics[k], spy_metrics[k]) for k in lstm_metrics}
lstm_vs_eq  = {k: excess(lstm_metrics[k], eq_metrics[k])  for k in lstm_metrics}

pct_metrics = {'Total Return', 'Annual Return', 'Annual Vol', 'Max Drawdown'}
strategies  = ['Dynamic LSTM', 'SPY Buy & Hold', 'Equal Weight']
metrics_all = [lstm_metrics, spy_metrics, eq_metrics]

header = f'{"Metric":<22s}' + ''.join(f'{s:>18s}' for s in strategies)
print(f'\n{header}')
print('-' * (22 + 18 * len(strategies)))
for metric in lstm_metrics:
    row = f'{metric:<22s}'
    for m in metrics_all:
        val = m[metric]
        row += f'{val:>17.1%} ' if metric in pct_metrics else f'{val:>17.4f} '
    print(row)

for label, diff in [('Excess vs SPY', lstm_vs_spy), ('Excess vs Equal Weight', lstm_vs_eq)]:
    print(f'\n{label}')
    print('-' * 40)
    for metric in ['Total Return', 'Annual Return', 'Sharpe', 'Max Drawdown']:
        val  = diff[metric]
        fmt  = f'{val:>+.1%}' if metric in pct_metrics else f'{val:>+.4f}'
        flag = 'OK' if val > 0 else 'CHECK'
        print(f'  {flag:<5s} {metric:<20s}: {fmt}')

print('\n-- Portfolio Shape --')
print(f'  Avg daily turnover      : {result.turnover.mean():.4f}')
print(f'  Avg active names        : {selected_diag["selected_names"].mean():.1f}')
print(f'  Avg effective names     : {selected_diag["effective_names"].mean():.1f}')
print(f'  Avg max weight          : {selected_diag["max_weight"].mean():.2%}')
print(f'  Concentration-on days   : {selected_diag["concentration_on"].mean():.1%}')
print(f'  Risk-on gate days       : {selected_diag["risk_on_gate"].mean():.1%}')
print(f'  Uncertain gate days     : {selected_diag["uncertain_gate"].mean():.1%}')
print('  Dynamic regime mix      :')
print(selected_diag['regime'].value_counts(normalize=True).to_string())

_comparison_metrics_cache = {
    'lstm_annual_return'          : lstm_metrics['Annual Return'],
    'lstm_sharpe'                 : lstm_metrics['Sharpe'],
    'lstm_max_drawdown'           : lstm_metrics['Max Drawdown'],
    'spy_annual_return'           : spy_metrics['Annual Return'],
    'spy_sharpe'                  : spy_metrics['Sharpe'],
    'eq_annual_return'            : eq_metrics['Annual Return'],
    'eq_sharpe'                   : eq_metrics['Sharpe'],
    'excess_annual_return_vs_spy' : lstm_vs_spy['Annual Return'],
    'excess_sharpe_vs_spy'        : lstm_vs_spy['Sharpe'],
    'excess_annual_return_vs_eq'  : lstm_vs_eq['Annual Return'],
    'excess_sharpe_vs_eq'         : lstm_vs_eq['Sharpe'],
    'avg_active_names'            : float(selected_diag['selected_names'].mean()),
    'avg_effective_names'         : float(selected_diag['effective_names'].mean()),
    'avg_max_weight'              : float(selected_diag['max_weight'].mean()),
    'concentration_day_rate'      : float(selected_diag['concentration_on'].mean()),
    'risk_on_day_rate'           : float(selected_diag['risk_on_gate'].mean()),
    'uncertain_day_rate'         : float(selected_diag['uncertain_gate'].mean()),
    'avg_selected_bottom_prob'   : float(selected_diag['avg_bottom_probability_selected'].mean()),
    'avg_selected_risk_quality'  : float(selected_diag['avg_risk_quality_selected'].mean()),
}

for regime_name, regime_share in selected_diag['regime'].value_counts(normalize=True).items():
    _comparison_metrics_cache[f'dynamic_regime_share_{regime_name}'] = float(regime_share)

for _, row in sleeve_summary.iterrows():
    prefix = f"sleeve_{row['portfolio']}"
    _comparison_metrics_cache[f'{prefix}_annual_return'] = float(row['annual_return'])
    _comparison_metrics_cache[f'{prefix}_sharpe'] = float(row['sharpe'])
    _comparison_metrics_cache[f'{prefix}_max_drawdown'] = float(row['max_drawdown'])

print('\n' + '=' * 80)
print('BENCHMARK + SLEEVE COMPARISON COMPLETE')
print('=' * 80)


In [ ]:
# ADVERSARIAL STRESS: try to find how the strategy loses money
print('=' * 80)
print('ADVERSARIAL STRESS: HOW DO WE LOSE MONEY?')
print('=' * 80)

stress_df = pd.DataFrame({
    'strategy_return': result.returns,
    'benchmark_return': spy_returns.reindex(result.returns.index).fillna(0.0),
    'turnover': result.turnover.reindex(result.returns.index).fillna(0.0),
})
stress_df['active_names'] = (aligned_weights > 1e-12).sum(axis=1).reindex(stress_df.index).ffill()
stress_df['max_weight'] = aligned_weights.max(axis=1).reindex(stress_df.index).ffill()
stress_df['effective_names'] = aligned_weights.apply(_effective_names, axis=1).reindex(stress_df.index).ffill()
stress_df['strategy_drawdown'] = result.nav / result.nav.cummax() - 1.0

worst_strategy_days = stress_df.nsmallest(10, 'strategy_return')
worst_benchmark_days = stress_df.nsmallest(10, 'benchmark_return')
high_turnover_days = stress_df.nlargest(10, 'turnover')

print('\nWorst strategy days')
display(worst_strategy_days.style.format('{:.2%}', subset=['strategy_return', 'benchmark_return', 'strategy_drawdown', 'max_weight']))

print('\nWorst benchmark days: did the strategy defend capital?')
benchmark_stress = worst_benchmark_days.assign(excess_vs_benchmark=lambda df: df['strategy_return'] - df['benchmark_return'])
display(benchmark_stress.style.format('{:.2%}', subset=['strategy_return', 'benchmark_return', 'excess_vs_benchmark', 'strategy_drawdown', 'max_weight']))



print('\nDynamic allocator regimes on worst strategy days')
regime_by_date = selected_diag.set_index('date')[['regime', 'top_confidence', 'top_risk_quality', 'predicted_spread_90_10']]
worst_with_regime = worst_strategy_days.join(regime_by_date, how='left')
display(worst_with_regime.style.format({
    'strategy_return': '{:.2%}',
    'benchmark_return': '{:.2%}',
    'strategy_drawdown': '{:.2%}',
    'max_weight': '{:.2%}',
    'top_confidence': '{:.2%}',
    'top_risk_quality': '{:.2%}',
    'predicted_spread_90_10': '{:.2%}',
}))

print('\nHighest turnover days')
display(high_turnover_days.style.format('{:.2%}', subset=['strategy_return', 'benchmark_return', 'turnover', 'strategy_drawdown', 'max_weight']))

stress_summary = {
    'worst_strategy_day': float(stress_df['strategy_return'].min()),
    'avg_return_on_worst_benchmark_10_days': float(worst_benchmark_days['strategy_return'].mean()),
    'avg_excess_on_worst_benchmark_10_days': float((worst_benchmark_days['strategy_return'] - worst_benchmark_days['benchmark_return']).mean()),
    'avg_active_names': float(stress_df['active_names'].mean()),
    'avg_effective_names': float(stress_df['effective_names'].mean()),
    'avg_max_weight': float(stress_df['max_weight'].mean()),
    'avg_turnover': float(stress_df['turnover'].mean()),
    'risk_on_share': float(selected_diag['risk_on_gate'].mean()),
    'uncertain_share': float(selected_diag['uncertain_gate'].mean()),
}

print('\nStress summary')
for k, v in stress_summary.items():
    fmt = f'{v:.2%}' if any(token in k for token in ['day', 'return', 'excess', 'weight', 'turnover']) else f'{v:.2f}'
    print(f'  {k:<42s}: {fmt}')

if '_comparison_metrics_cache' in dir():
    _comparison_metrics_cache.update({f'stress_{k}': v for k, v in stress_summary.items()})
print('=' * 80)


------------------------------------------------------------------------------------------

## 12. Write Artifacts (QuantStats report + parquet files)

In [ ]:
artifact_paths = write_backtest_artifacts(result, artifact_dir)

for key, path in artifact_paths.items():
    exists = '✓' if Path(path).exists() else '✗'
    print(f'  {exists} {key:<20s}: {path}')

## 13. Log to MLflow as `Minho_Baseline_LSTM`

In [ ]:
import mlflow
from portfolio_toolkit.tracking import log_model_submission

mlflow_layout = init_mlflow(repo_root)
print('Tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name     = RUN_NAME,
    dataset_name = dataset,        # ← pass the object, not dataset.name
    tags={
        'author'        : 'Minho',
        'model_family'  : MODEL_FAMILY,
        'workflow'      : 'ranking_confidence_lstm_v4_dynamic_risk',
        'horizon'       : str(HORIZON),
        'dataset'       : dataset.name,
        'project'       : 'risk_adjusted_portfolio_optimization',
    },
    repo_root=repo_root,
):
    # Params
    mlflow.log_params({
        'run_name'            : RUN_NAME,
        'dataset'             : dataset.name if hasattr(dataset, 'name') else str(DATASET_NAME),
        'horizon'             : HORIZON,
        'seq_len'             : SEQ_LEN,
        'hidden_size'         : HIDDEN_SIZE,
        'num_layers'          : NUM_LAYERS,
        'dropout'             : DROPOUT,
        'lr'                  : LR,
        'epochs'              : EPOCHS,
        'batch_size'          : BATCH_SIZE,
        'n_features'          : len(ALL_FEATURES),
        'custom_features'     : ','.join(CUSTOM_FEATURES),
        'regime_features'     : ','.join(REGIME_FEATURES),
        'n_cs_features'       : len(CS_FEATURES),
        'portfolio_builder'   : 'confidence_dynamic_risk_budget',
        'best_val_mse'        : round(float(best_val), 8),
        'target_type'         : TARGET_TYPE,
        'rebalance_freq'      : REBALANCE_FREQ,
        'rebalance_frequency' : 'every_5_trading_days',   # ← string format Adam requires
        'max_weight_cap'      : MAX_WEIGHT,
        'rank_loss_weight'    : RANK_LOSS_WEIGHT,
        'top_quintile_weight' : TOP_QUINTILE_WEIGHT,
        'vol_loss_weight'     : VOL_LOSS_WEIGHT,
        'downside_loss_weight': DOWNSIDE_LOSS_WEIGHT,
        'bottom_quintile_weight': BOTTOM_QUINTILE_WEIGHT,
        'worst_path_loss_weight': WORST_PATH_LOSS_WEIGHT,
        'aux_alpha_loss_weight' : AUX_ALPHA_LOSS_WEIGHT,
        'sleeve_sizes'        : ','.join(map(str, SLEEVE_SIZES)),
        'dynamic_top_n'      : json.dumps(DYNAMIC_TOP_N),
        'aux_horizons'       : ','.join(map(str, AUX_HORIZONS)),
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'high_confidence_threshold': HIGH_CONFIDENCE_THRESHOLD,
        'low_confidence_threshold' : LOW_CONFIDENCE_THRESHOLD,
        'spread_threshold'        : SPREAD_THRESHOLD,
        'risk_quality_threshold'  : RISK_QUALITY_THRESHOLD,
        'defensive_max_weight'    : DEFENSIVE_MAX_WEIGHT,
        'risk_on_max_weight'      : RISK_ON_MAX_WEIGHT,
        'loser_rank_cutoff'       : LOSER_RANK_CUTOFF,
        'bottom_prob_cutoff'      : BOTTOM_PROB_CUTOFF,
        'min_active_names'        : MIN_ACTIVE_NAMES,
        'no_trade_band'       : NO_TRADE_BAND,
        'training_cutoff'     : str(TRAINING_CUTOFF),
        'softmax_temperature' : SOFTMAX_TEMPERATURE,
        'risk_aversion'       : RISK_AVERSION,
        'downside_aversion'   : DOWNSIDE_AVERSION,
    })

    # Validation metrics
    mlflow.log_metric('val_mse',  round(float(best_val), 8))
    mlflow.log_metric('val_rmse', round(float(best_val ** 0.5), 8))

    # Diagnostic metrics
    if '_diagnostic_metrics' in dir():
        mlflow.log_metrics(_diagnostic_metrics)

    # Backtest metrics + report artifacts
    log_predictions(predictions)
    log_portfolio(clean_portfolio)
    log_backtest(result)

    # Benchmark comparison metrics
    if '_comparison_metrics_cache' in dir():
        mlflow.log_metrics(_comparison_metrics_cache)
        print('Benchmark comparison metrics logged.')

    # ── Model submission bundle (Adam's required format) ──────────────────
    notebook_path = repo_root / 'baseline_lstm.ipynb'

    log_model_submission(
        {'torch_model': str(checkpoint_path)},
        model_name          = RUN_NAME,
        model_family        = 'torch',
        feature_names       = ALL_FEATURES,
        target              = TARGET_COL,
        horizon             = HORIZON,
        rebalance_frequency = 'every_5_trading_days',
        preprocessing       = {'scaler': 'train_mean_std'},
        model_config        = {
            'architecture'      : 'RankingConfidenceLSTM',
            'input_dim'         : len(ALL_FEATURES),
            'hidden_dim'        : HIDDEN_SIZE,
            'num_layers'        : NUM_LAYERS,
            'portfolio_builder' : 'build_confidence_portfolio',
            'aux_horizons'       : AUX_HORIZONS,
            'dynamic_top_n'      : DYNAMIC_TOP_N,
            'required_functions': ['build_model_features', 'predict_from_prices', 'build_confidence_portfolio'],
        },
        source_files = [str(notebook_path)],
    )
    print('Model submission bundle logged to MLflow.')

print(f'MLflow run "{RUN_NAME}" logged successfully.')
print('Artifacts visible in MLflow UI under: Artifacts → model_submission/')

In [ ]:
# ── Reproducibility verification ──────────────────────────────────
import torch
import numpy as np
import pandas as pd

def load_model_from_artifacts(checkpoint_dir: Path) -> tuple:
    """
    Reconstruct the trained model + preprocessing from the bundled checkpoint.
    """
    checkpoint_path = checkpoint_dir / 'lstm_checkpoint.pt'
    bundle = torch.load(checkpoint_path, map_location='cpu')

    cfg = bundle['model_config']
    reconstructed_model = BaselineLSTM(
        input_size   = cfg['input_dim'],
        hidden_size  = cfg['hidden_dim'],
        num_layers   = cfg['num_layers'],
        dropout      = cfg['dropout'],
        aux_horizons = cfg.get('aux_horizons', []),
    )
    reconstructed_model.load_state_dict(bundle['state_dict'])
    reconstructed_model.eval()

    train_mean_loaded = pd.Series(bundle['preprocessing']['means'])
    train_std_loaded  = pd.Series(bundle['preprocessing']['stds'])

    return reconstructed_model, train_mean_loaded, train_std_loaded, bundle


# ── Run the verification ───────────────────────────────────────────────────────
print('Loading model from bundled checkpoint (simulating fresh session) ...')
reconstructed_model, train_mean_loaded, train_std_loaded, loaded_bundle =     load_model_from_artifacts(checkpoint_dir)

print(f'  Architecture  : {loaded_bundle["model_config"]}')
print(f'  Feature count : {len(loaded_bundle["feature_names"])}')
print(f'  Horizon       : {loaded_bundle["horizon"]}')
print(f'  Aux horizons  : {loaded_bundle.get("aux_horizons", [])}')
print(f'  Rebal freq    : {loaded_bundle["rebalance_frequency"]}')
print(f'  Source        : {loaded_bundle["source_notebook"]}')

# ── Reproduce predictions ──────────────────────────────────────────────────────
print('\nReproducing predictions from reconstructed model ...')
reproduced_predictions = predict_from_prices(
    model         = reconstructed_model,
    prices        = prices,
    train_mean    = train_mean_loaded,
    train_std     = train_std_loaded,
    feature_names = loaded_bundle['feature_names'],
    seq_len       = loaded_bundle['data']['seq_len'],
    horizon       = loaded_bundle['horizon'],
    tickers       = TRADABLE_TICKERS,
)

reproduced_predictions = reproduced_predictions[
    reproduced_predictions['date'].isin(test_dates)
].reset_index(drop=True)

rebal_dates_repro = set(pd.DatetimeIndex(
    sorted(reproduced_predictions['date'].unique())
)[::REBALANCE_FREQ])
reproduced_predictions = reproduced_predictions[
    reproduced_predictions['date'].isin(rebal_dates_repro)
].reset_index(drop=True)

# ── Verify match ───────────────────────────────────────────────────────────────
print('\nVerifying reproduced predictions match originals ...')
orig  = predictions.set_index(['date', 'ticker']).sort_index()
repro = reproduced_predictions.set_index(['date', 'ticker']).sort_index()

assert orig.shape == repro.shape,     f'Shape mismatch: original {orig.shape} vs reproduced {repro.shape}'

for col in ['expected_return', 'confidence', 'bottom_probability', 'expected_volatility', 'expected_downside']:
    if col in orig.columns and col in repro.columns:
        max_diff = float((orig[col] - repro[col]).abs().max())
        assert max_diff < 1e-5, f'{col} differs by up to {max_diff:.2e}'
        print(f'  {col:<22s}: max diff {max_diff:.2e}')

print(f'  Rows compared          : {len(orig):,}')
print()
print('Reproducibility check PASSED.')
print('Adam can reconstruct identical v4 predictions from MLflow artifacts alone.')


In [ ]:
print(f'orig shape: {orig.shape}, repro shape: {repro.shape}')
print('Reproducibility assertions completed in the previous cell.')


## 14. Final Checks

In [ ]:
from IPython.display import display

# Sanity assertions
assert {'total_return', 'annual_return', 'sharpe', 'max_drawdown'}.issubset(result.metrics)
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(artifact_paths['quantstats_report']).exists()
assert {'date', 'ticker', 'horizon', 'expected_return'}.issubset(predictions.columns)
assert 'price_accel' in frame.columns
assert {'confidence', 'uncertainty'}.issubset(predictions.columns)
assert selected_diag['selected_names'].mean() >= MIN_ACTIVE_NAMES
assert spec.train_end <= TRAINING_CUTOFF
assert spec.val_end <= TRAINING_CUTOFF
assert spec.test_start > TRAINING_CUTOFF

print('All checks passed. Ranking-confidence notebook ran end to end successfully.')
print()
print('Key metrics:')
for k in ['annual_return', 'sharpe', 'max_drawdown', 'average_turnover']:
    print(f'  {k:<20s}: {result.metrics[k]:.4f}')

print()
print()
print('Portfolio shape:')
print(f'  avg_active_names    : {selected_diag["selected_names"].mean():.1f}')
print(f'  avg_effective_names : {selected_diag["effective_names"].mean():.1f}')
print(f'  avg_max_weight      : {selected_diag["max_weight"].mean():.2%}')
print()
display(result.nav.tail(3).to_frame('nav'))

## Four-Regime Proxy Dataset Backtests

This section is self-contained: it bootstraps the repo, loads Minho's downloaded Run `4403181aa77c465b85c40ec6a237befa` v4 dynamic-risk checkpoint from `MODELS/Minho/lstm_checkpoint_v4_dynamic_risk.pt`, rebuilds the v4 feature and portfolio pipeline, runs the four proxy-regime backtests, summarizes the `$4.8M` bankroll, and logs the results to MLflow. You can run these cells from a fresh kernel without running the training cells above.


In [1]:
import json
import os
import sys
import warnings
from datetime import timedelta
from pathlib import Path

warnings.filterwarnings("ignore")

import mlflow
import numpy as np
import pandas as pd
import torch
import yfinance as yf
from IPython.display import display
from torch import nn


def _minho_v4_find_repo_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "configs" / "datasets.toml").exists() and (candidate / "src" / "portfolio_toolkit").exists():
            return candidate
    raise RuntimeError("Could not locate the Portfolio-Optimizer repo root from the current working directory.")


repo_root = _minho_v4_find_repo_root()
os.chdir(repo_root)
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from portfolio_toolkit import (  # noqa: E402
    PortfolioWeights,
    backtest_weights,
    build_features,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    log_predictions,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    write_backtest_artifacts,
)

MINHO_V4_SOURCE_RUN_ID = "4403181aa77c465b85c40ec6a237befa"
MINHO_V4_SOURCE_RUN_NAME = "Minho_RankingConfidence_LSTM_v4_dynamic_risk"
MINHO_V4_MODEL_DIR = repo_root / "MODELS" / "Minho"
MINHO_V4_NOTEBOOK_PATH = MINHO_V4_MODEL_DIR / "baseline_lstm_v4_dynamic_risk.ipynb"
MINHO_V4_CHECKPOINT_PATH = MINHO_V4_MODEL_DIR / "lstm_checkpoint_v4_dynamic_risk.pt"
MINHO_V4_MANIFEST_PATH = MINHO_V4_MODEL_DIR / "minho_v4_dynamic_risk_manifest.json"
if not MINHO_V4_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing downloaded Minho v4 checkpoint: {MINHO_V4_CHECKPOINT_PATH}")
if not MINHO_V4_NOTEBOOK_PATH.exists():
    raise FileNotFoundError(f"Missing downloaded Minho v4 notebook: {MINHO_V4_NOTEBOOK_PATH}")

try:
    MINHO_V4_CHECKPOINT = torch.load(MINHO_V4_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
except TypeError:
    MINHO_V4_CHECKPOINT = torch.load(MINHO_V4_CHECKPOINT_PATH, map_location="cpu")
MINHO_V4_MANIFEST = json.loads(MINHO_V4_MANIFEST_PATH.read_text(encoding="utf-8")) if MINHO_V4_MANIFEST_PATH.exists() else {}

MINHO_V4_MODEL_NAME = str(MINHO_V4_CHECKPOINT.get("run_name", MINHO_V4_SOURCE_RUN_NAME))
MINHO_V4_FEATURE_NAMES = list(MINHO_V4_CHECKPOINT["feature_names"])
MINHO_V4_HORIZON = int(MINHO_V4_CHECKPOINT.get("horizon", MINHO_V4_MANIFEST.get("horizon", 5)))
MINHO_V4_SEQ_LEN = int(MINHO_V4_CHECKPOINT.get("data", {}).get("seq_len", 20))
MINHO_V4_AUX_HORIZONS = list(MINHO_V4_CHECKPOINT.get("aux_horizons", MINHO_V4_CHECKPOINT.get("model_config", {}).get("aux_horizons", [10, 20])))
MINHO_V4_REBALANCE_FREQUENCY = str(MINHO_V4_CHECKPOINT.get("rebalance_frequency", MINHO_V4_MANIFEST.get("rebalance_frequency", "every_5_trading_days")))
MINHO_V4_REBALANCE_EVERY_N_DAYS = int(MINHO_V4_REBALANCE_FREQUENCY.split("_")[1]) if MINHO_V4_REBALANCE_FREQUENCY.startswith("every_") else 5
MINHO_V4_PREPROCESSING = dict(MINHO_V4_CHECKPOINT.get("preprocessing", {}))
MINHO_V4_TRAIN_MEAN = pd.to_numeric(pd.Series(MINHO_V4_PREPROCESSING.get("means", {})).reindex(MINHO_V4_FEATURE_NAMES), errors="coerce").fillna(0.0)
MINHO_V4_TRAIN_STD = pd.to_numeric(pd.Series(MINHO_V4_PREPROCESSING.get("stds", {})).reindex(MINHO_V4_FEATURE_NAMES), errors="coerce").replace(0.0, 1.0).fillna(1.0)

MINHO_V4_PORTFOLIO_CONFIG = dict(MINHO_V4_CHECKPOINT.get("portfolio_config", {}))
DYNAMIC_TOP_N = MINHO_V4_PORTFOLIO_CONFIG.get("dynamic_top_n", {"high": 15, "normal": 28, "low": 45})
SLEEVE_SIZES = MINHO_V4_PORTFOLIO_CONFIG.get("sleeve_sizes", [12, 18, 29, 45])
CONFIDENCE_THRESHOLD = float(MINHO_V4_PORTFOLIO_CONFIG.get("confidence_threshold", 0.62))
HIGH_CONFIDENCE_THRESHOLD = float(MINHO_V4_PORTFOLIO_CONFIG.get("high_confidence_threshold", 0.70))
LOW_CONFIDENCE_THRESHOLD = float(MINHO_V4_PORTFOLIO_CONFIG.get("low_confidence_threshold", 0.54))
SPREAD_THRESHOLD = float(MINHO_V4_PORTFOLIO_CONFIG.get("spread_threshold", 0.015))
RISK_QUALITY_THRESHOLD = float(MINHO_V4_PORTFOLIO_CONFIG.get("risk_quality_threshold", 0.55))
MAX_WEIGHT = float(MINHO_V4_PORTFOLIO_CONFIG.get("max_weight", 0.040))
DEFENSIVE_MAX_WEIGHT = float(MINHO_V4_PORTFOLIO_CONFIG.get("defensive_max_weight", 0.025))
RISK_ON_MAX_WEIGHT = float(MINHO_V4_PORTFOLIO_CONFIG.get("risk_on_max_weight", 0.060))
MIN_ACTIVE_NAMES = int(MINHO_V4_PORTFOLIO_CONFIG.get("min_active_names", 12))
LOSER_RANK_CUTOFF = float(MINHO_V4_PORTFOLIO_CONFIG.get("loser_rank_cutoff", 0.12))
BOTTOM_PROB_CUTOFF = float(MINHO_V4_PORTFOLIO_CONFIG.get("bottom_prob_cutoff", 0.62))
NO_TRADE_BAND = float(MINHO_V4_PORTFOLIO_CONFIG.get("no_trade_band", 0.0050))
SOFTMAX_TEMPERATURE = 0.20
DEFENSIVE_TEMPERATURE = 0.45
RISK_AVERSION = 1.75
DOWNSIDE_AVERSION = 2.25
TAIL_RISK_AVERSION = 1.75

MINHO_V4_REGIME_BACKTEST_DATASETS = [
    "regime_modern_tech_gain_2022_2026",
    "regime_financial_crisis_loss_2005_2010",
    "regime_nineties_volatility_1995_1999",
    "regime_oil_pre2014_energy_2010_2013",
]
MINHO_V4_REGIME_BACKTEST_ALLOCATION = 1_200_000.0
MINHO_V4_REGIME_BACKTEST_TOTAL_BANKROLL = MINHO_V4_REGIME_BACKTEST_ALLOCATION * len(MINHO_V4_REGIME_BACKTEST_DATASETS)
MINHO_V4_REGIME_BACKTEST_OUTPUT_DIR = repo_root / "runs" / "minho_v4_dynamic_risk_four_regime_proxy_backtests"
MINHO_V4_REGIME_BACKTEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", repo_root)
print("Model:", MINHO_V4_MODEL_NAME)
print("Checkpoint:", MINHO_V4_CHECKPOINT_PATH)
print("Feature count:", len(MINHO_V4_FEATURE_NAMES))
print("Horizon/rebalance:", MINHO_V4_HORIZON, MINHO_V4_REBALANCE_FREQUENCY)
print("Four-regime bankroll: ${:,.0f}".format(MINHO_V4_REGIME_BACKTEST_TOTAL_BANKROLL))


Repo root: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer
Model: Minho_RankingConfidence_LSTM_v4_dynamic_risk
Checkpoint: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer/MODELS/Minho/lstm_checkpoint_v4_dynamic_risk.pt
Feature count: 78
Horizon/rebalance: 5 every_5_trading_days
Four-regime bankroll: $4,800,000


In [2]:
MINHO_V4_BASE_FEATURES = [
    "return_1d", "return_5d", "return_10d", "return_20d", "return_60d",
    "vol_5d", "vol_10d", "vol_20d", "vol_60d",
    "downside_vol_20d", "upside_vol_20d", "atr_14",
    "momentum_5d", "momentum_10d", "momentum_20d", "momentum_60d", "momentum_120d",
    "rsi_14", "macd_hist", "bollinger_z_20d",
    "beta_20d_spy", "beta_60d_spy",
    "excess_return_5d_vs_spy", "excess_return_20d_vs_spy", "excess_return_60d_vs_spy",
    "relative_momentum_20d_vs_spy",
    "volume_zscore_20d", "volume_zscore_60d", "dollar_volume_ratio_20d",
    "price_to_sma_20d", "price_to_sma_50d", "price_to_sma_200d",
    "intraday_range", "close_open_gap", "close_location_in_range",
    "distance_to_20d_high", "distance_to_20d_low",
    "distance_to_60d_high", "distance_to_60d_low",
    "skew_20d", "kurtosis_20d",
]


def _minho_v4_normalize_yfinance_frame(frame: pd.DataFrame, ticker: str) -> pd.DataFrame:
    normalized = frame.copy()
    if isinstance(normalized.columns, pd.MultiIndex):
        normalized.columns = normalized.columns.get_level_values(0)
    normalized.columns.name = None
    if "Adj Close" not in normalized.columns and "Close" in normalized.columns:
        normalized["Adj Close"] = normalized["Close"]
    normalized = normalized.reset_index().rename(
        columns={
            "Date": "date", "Datetime": "date", "Open": "open", "High": "high", "Low": "low",
            "Close": "close", "Adj Close": "adj_close", "Volume": "volume",
        }
    )
    normalized["ticker"] = ticker.upper()
    normalized = normalized.loc[:, ["date", "ticker", "open", "high", "low", "close", "adj_close", "volume"]]
    normalized["date"] = pd.to_datetime(normalized["date"], utc=True).dt.tz_localize(None)
    for column in ["open", "high", "low", "close", "adj_close", "volume"]:
        normalized[column] = pd.to_numeric(normalized[column], errors="coerce")
    return normalized.dropna(subset=["date", "adj_close"]).sort_values(["ticker", "date"]).reset_index(drop=True)


def _minho_v4_download_context_ticker(ticker: str, spec_obj) -> pd.DataFrame:
    start = pd.Timestamp(spec_obj.start_date).date().isoformat()
    end = (pd.Timestamp(spec_obj.end_date).date() + timedelta(days=1)).isoformat()
    downloaded = yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False, threads=False)
    if downloaded.empty:
        raise ValueError(f"No yfinance rows downloaded for required context ticker {ticker} in {spec_obj.identifier}.")
    return _minho_v4_normalize_yfinance_frame(downloaded, ticker)


def _minho_v4_prices_with_spy_context(prices_frame: pd.DataFrame, spec_obj) -> pd.DataFrame:
    prices_for_features = prices_frame.copy()
    prices_for_features["date"] = pd.to_datetime(prices_for_features["date"], utc=True).dt.tz_localize(None)
    prices_for_features["ticker"] = prices_for_features["ticker"].astype(str).str.upper()
    if "SPY" not in set(prices_for_features["ticker"]):
        spy_context = _minho_v4_download_context_ticker("SPY", spec_obj)
        prices_for_features = pd.concat([prices_for_features, spy_context], ignore_index=True)
    return prices_for_features.sort_values(["ticker", "date"]).reset_index(drop=True)


def _add_market_regime_features(prices: pd.DataFrame, frame: pd.DataFrame) -> pd.DataFrame:
    panel = prices.sort_values(["ticker", "date"]).copy()
    panel["ret_1d"] = panel.groupby("ticker")["adj_close"].pct_change()
    panel["ret_5d"] = panel.groupby("ticker")["adj_close"].pct_change(5)
    panel["ret_20d"] = panel.groupby("ticker")["adj_close"].pct_change(20)

    spy = panel.loc[panel["ticker"] == "SPY", ["date", "adj_close", "ret_1d", "ret_5d", "ret_20d"]].copy()
    if spy.empty:
        raise ValueError("SPY context rows are required for Minho v4 SPY-relative features.")
    spy = spy.sort_values("date")
    spy["spy_vol_20d"] = spy["ret_1d"].rolling(20, min_periods=20).std(ddof=0)
    spy["spy_vol_60d"] = spy["ret_1d"].rolling(60, min_periods=60).std(ddof=0)
    spy["spy_sma_20d"] = spy["adj_close"].rolling(20, min_periods=20).mean()
    spy["spy_sma_60d"] = spy["adj_close"].rolling(60, min_periods=60).mean()
    spy["spy_trend_20d"] = spy["adj_close"] / spy["spy_sma_20d"] - 1.0
    spy["spy_trend_60d"] = spy["adj_close"] / spy["spy_sma_60d"] - 1.0
    spy["spy_drawdown_60d"] = spy["adj_close"] / spy["adj_close"].rolling(60, min_periods=20).max() - 1.0
    spy["spy_rebound_5d"] = spy["ret_5d"].clip(lower=0.0)

    tradable_panel = panel.loc[panel["ticker"] != "SPY"].copy()
    breadth = tradable_panel.groupby("date").agg(
        universe_breadth_20d=("ret_20d", lambda s: float((s > 0).mean())),
        universe_dispersion_5d=("ret_5d", "std"),
        universe_dispersion_20d=("ret_20d", "std"),
        universe_median_return_5d=("ret_5d", "median"),
        universe_median_return_20d=("ret_20d", "median"),
    ).reset_index()

    regime = spy[[
        "date", "ret_5d", "ret_20d", "spy_vol_20d", "spy_vol_60d",
        "spy_trend_20d", "spy_trend_60d", "spy_drawdown_60d", "spy_rebound_5d",
    ]].rename(columns={"ret_5d": "spy_return_5d", "ret_20d": "spy_return_20d"})
    regime = regime.merge(breadth, on="date", how="left")
    regime["high_vol_regime"] = (regime["spy_vol_20d"] >= regime["spy_vol_20d"].rolling(252, min_periods=60).median()).astype(float)
    regime["drawdown_regime"] = (regime["spy_drawdown_60d"] <= -0.08).astype(float)
    regime["rebound_regime"] = ((regime["spy_return_5d"] > 0.03) & (regime["spy_drawdown_60d"] > -0.12)).astype(float)
    return frame.merge(regime, on="date", how="left")


def _add_cross_sectional_features(frame: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    out = frame.copy()
    grouped = out.groupby("date", sort=False)
    for col in base_cols:
        if col not in out.columns:
            continue
        out[f"{col}_cs_rank"] = grouped[col].rank(pct=True, method="average")
        mean = grouped[col].transform("mean")
        std = grouped[col].transform("std").replace(0.0, np.nan)
        out[f"{col}_cs_z"] = (out[col] - mean) / std
    return out


def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    prices_local = prices.copy()
    prices_local["date"] = pd.to_datetime(prices_local["date"], utc=True).dt.tz_localize(None)
    prices_local["ticker"] = prices_local["ticker"].astype(str).str.upper()
    prices_local = prices_local.sort_values(["ticker", "date"]).reset_index(drop=True)
    feature_frame = build_features(prices_local, feature_names=MINHO_V4_BASE_FEATURES)

    panel = prices_local.sort_values(["ticker", "date"]).copy()
    ret5 = panel.groupby("ticker")["adj_close"].pct_change(5)
    ret20 = panel.groupby("ticker")["adj_close"].pct_change(20)
    ret60 = panel.groupby("ticker")["adj_close"].pct_change(60)
    daily_ret = panel.groupby("ticker")["adj_close"].pct_change()
    vol20 = daily_ret.groupby(panel["ticker"]).transform(lambda s: s.rolling(20, min_periods=20).std(ddof=0))
    vol60 = daily_ret.groupby(panel["ticker"]).transform(lambda s: s.rolling(60, min_periods=60).std(ddof=0))

    custom_df = panel[["date", "ticker"]].copy()
    custom_df["price_accel"] = (ret5 - ret20).values
    custom_df["momentum_reversal_20_60"] = (ret20 - ret60).values
    custom_df["vol_compression_20_60"] = (vol20 / vol60.replace(0.0, np.nan)).values

    frame = feature_frame.merge(custom_df, on=["date", "ticker"], how="left")
    frame = _add_market_regime_features(prices_local, frame)
    frame = _add_cross_sectional_features(
        frame,
        [
            "momentum_20d", "momentum_60d", "excess_return_20d_vs_spy",
            "relative_momentum_20d_vs_spy", "vol_20d", "downside_vol_20d",
            "volume_zscore_20d", "price_accel", "distance_to_20d_high",
        ],
    )
    for column in MINHO_V4_FEATURE_NAMES:
        if column not in frame.columns:
            frame[column] = np.nan
    return frame[["date", "ticker"] + MINHO_V4_FEATURE_NAMES].sort_values(["ticker", "date"]).reset_index(drop=True)


class RankingConfidenceLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2, aux_horizons=None):
        super().__init__()
        self.aux_horizons = list(aux_horizons or [])
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.alpha_head = nn.Linear(hidden_size, 1)
        self.confidence_head = nn.Linear(hidden_size, 1)
        self.bottom_head = nn.Linear(hidden_size, 1)
        self.volatility_head = nn.Linear(hidden_size, 1)
        self.downside_head = nn.Linear(hidden_size, 1)
        self.worst_path_head = nn.Linear(hidden_size, 1)
        self.aux_alpha_heads = nn.ModuleDict({str(h): nn.Linear(hidden_size, 1) for h in self.aux_horizons})

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        h = self.dropout(self.norm(h_n[-1]))
        outputs = {
            "alpha": self.alpha_head(h).squeeze(-1),
            "confidence_logit": self.confidence_head(h).squeeze(-1),
            "bottom_logit": self.bottom_head(h).squeeze(-1),
            "volatility": self.volatility_head(h).squeeze(-1),
            "downside": self.downside_head(h).squeeze(-1),
            "worst_path": self.worst_path_head(h).squeeze(-1),
        }
        for horizon, head in self.aux_alpha_heads.items():
            outputs[f"alpha_{horizon}d"] = head(h).squeeze(-1)
        return outputs


BaselineLSTM = RankingConfidenceLSTM


def _minho_v4_load_model_from_checkpoint() -> RankingConfidenceLSTM:
    cfg = dict(MINHO_V4_CHECKPOINT["model_config"])
    model = RankingConfidenceLSTM(
        input_size=int(cfg.get("input_dim", len(MINHO_V4_FEATURE_NAMES))),
        hidden_size=int(cfg.get("hidden_dim", 64)),
        num_layers=int(cfg.get("num_layers", 2)),
        dropout=float(cfg.get("dropout", 0.2)),
        aux_horizons=list(cfg.get("aux_horizons", MINHO_V4_AUX_HORIZONS)),
    )
    model.load_state_dict(MINHO_V4_CHECKPOINT["state_dict"])
    model.eval()
    return model


def _model_outputs(model: nn.Module, x_tensor: torch.Tensor) -> dict[str, torch.Tensor]:
    outputs = model(x_tensor)
    if isinstance(outputs, dict):
        return outputs
    return {"alpha": outputs}


def predict_from_prices(
    model: nn.Module,
    prices: pd.DataFrame,
    train_mean: pd.Series,
    train_std: pd.Series,
    feature_names: list[str],
    seq_len: int,
    horizon: int,
    dates=None,
    tickers=None,
) -> pd.DataFrame:
    frame = build_model_features(prices).replace([np.inf, -np.inf], np.nan)
    frame = frame.dropna(subset=feature_names).reset_index(drop=True)
    if tickers is not None:
        wanted_tickers = [str(ticker).upper() for ticker in tickers]
        frame = frame.loc[frame["ticker"].isin(wanted_tickers)].reset_index(drop=True)
    if frame.empty:
        raise ValueError("No Minho v4 feature rows remain after cleaning/filtering.")

    X_scaled = ((frame[feature_names] - train_mean.reindex(feature_names)) / train_std.reindex(feature_names)).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X_scaled = X_scaled.to_numpy(dtype=np.float32)
    X_seqs, meta = [], []
    for ticker, grp in frame.groupby("ticker", sort=False):
        idx = grp.index.tolist()
        dates_grp = grp["date"].tolist()
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx)):
            rows = idx[end - seq_len + 1 : end + 1]
            X_seqs.append(X_scaled[rows])
            meta.append((dates_grp[end], ticker))
    if not X_seqs:
        raise ValueError("No LSTM sequences could be built; not enough rows per ticker.")

    model.eval()
    device = next(model.parameters()).device
    alpha_list, confidence_list, bottom_list, vol_list, downside_list, worst_path_list = [], [], [], [], [], []
    aux_lists = {h: [] for h in MINHO_V4_AUX_HORIZONS}
    with torch.no_grad():
        for i in range(0, len(X_seqs), 2048):
            x_tensor = torch.as_tensor(np.stack(X_seqs[i:i + 2048]), dtype=torch.float32, device=device)
            outputs = _model_outputs(model, x_tensor)
            alpha_list.append(outputs["alpha"].detach().cpu().numpy())
            confidence_list.append(torch.sigmoid(outputs["confidence_logit"]).detach().cpu().numpy())
            bottom_list.append(torch.sigmoid(outputs["bottom_logit"]).detach().cpu().numpy())
            vol_list.append(torch.nn.functional.softplus(outputs["volatility"]).detach().cpu().numpy())
            downside_list.append(torch.nn.functional.softplus(outputs["downside"]).detach().cpu().numpy())
            worst_path_list.append((-torch.nn.functional.softplus(outputs["worst_path"])).detach().cpu().numpy())
            for h in MINHO_V4_AUX_HORIZONS:
                key = f"alpha_{h}d"
                if key in outputs:
                    aux_lists[h].append(outputs[key].detach().cpu().numpy())

    pred_dates, pred_tickers = zip(*meta)
    predictions = pd.DataFrame({
        "date": pd.to_datetime(list(pred_dates)),
        "ticker": list(pred_tickers),
        "horizon": horizon,
        "expected_return": np.concatenate(alpha_list).astype(float),
    })
    predictions["expected_alpha"] = predictions["expected_return"]
    predictions["confidence"] = np.concatenate(confidence_list).astype(float)
    predictions["uncertainty"] = 1.0 - predictions["confidence"]
    predictions["bottom_probability"] = np.concatenate(bottom_list).astype(float)
    predictions["expected_volatility"] = np.concatenate(vol_list).astype(float)
    predictions["expected_downside"] = np.concatenate(downside_list).astype(float)
    predictions["expected_worst_path"] = np.concatenate(worst_path_list).astype(float)
    for h, values in aux_lists.items():
        if values:
            predictions[f"expected_alpha_{h}d"] = np.concatenate(values).astype(float)
    predictions = predictions.drop_duplicates(["date", "ticker", "horizon"], keep="last")
    if dates is not None:
        wanted_dates = pd.DatetimeIndex(pd.to_datetime(dates, utc=True).tz_localize(None)).normalize()
        predictions = predictions.loc[predictions["date"].isin(wanted_dates)]
    return predictions.sort_values(["date", "ticker"]).reset_index(drop=True)


minho_v4_model = _minho_v4_load_model_from_checkpoint()
print("Minho v4 feature, model, and prediction helpers are ready.")


Minho v4 feature, model, and prediction helpers are ready.


In [3]:
def _effective_names(weights: pd.Series) -> float:
    active = weights[weights > 0].astype(float)
    if active.empty:
        return 0.0
    return float(1.0 / np.square(active).sum())


def _project_capped_simplex(weights: pd.Series, max_weight: float) -> pd.Series:
    w = weights.clip(lower=0.0).astype(float).copy()
    if max_weight <= 0:
        raise ValueError("max_weight must be positive")
    if max_weight * len(w) < 1.0 - 1e-12:
        raise ValueError("max_weight is too small for the number of active names")
    if w.sum() <= 0:
        w[:] = 1.0
    result = pd.Series(0.0, index=w.index, dtype=float)
    remaining = w.index.copy()
    residual = 1.0
    for _ in range(len(w) + 1):
        if len(remaining) == 0 or residual <= 1e-15:
            break
        base = w.loc[remaining]
        alloc = pd.Series(residual / len(remaining), index=remaining) if base.sum() <= 0 else base / base.sum() * residual
        over = alloc > max_weight
        if not over.any():
            result.loc[remaining] = alloc
            residual = 0.0
            break
        capped_names = alloc.index[over]
        result.loc[capped_names] = max_weight
        residual -= max_weight * len(capped_names)
        remaining = alloc.index[~over]
    slack = 1.0 - result.sum()
    if abs(slack) > 1e-10:
        capacity = (max_weight - result).clip(lower=0.0)
        if capacity.sum() > 0:
            result += capacity / capacity.sum() * slack
    result = result.clip(lower=0.0, upper=max_weight)
    if abs(result.sum() - 1.0) > 1e-8:
        under = result < max_weight - 1e-12
        if under.any() and result.loc[under].sum() > 0:
            result.loc[under] += (1.0 - result.sum()) * (result.loc[under] / result.loc[under].sum())
    return result.fillna(0.0)


def _softmax_weights(scores: pd.Series, temperature: float) -> pd.Series:
    values = scores.astype(float).replace([np.inf, -np.inf], np.nan).fillna(scores.median())
    values = values - values.max()
    exp_values = np.exp(values / max(float(temperature), 1e-6))
    return pd.Series(exp_values, index=scores.index) / exp_values.sum()


def _apply_no_trade_band(current: pd.Series, previous: pd.Series | None, band: float) -> pd.Series:
    if previous is None or band <= 0:
        return current
    previous = previous.reindex(current.index).fillna(0.0)
    adjusted = current.copy()
    keep = (current - previous).abs() < band
    adjusted[keep] = previous[keep]
    if adjusted.sum() <= 0:
        return current
    return adjusted / adjusted.sum()


def _rank01(series: pd.Series, ascending: bool = True) -> pd.Series:
    return series.astype(float).rank(pct=True, method="average", ascending=ascending).fillna(0.5)


def _minho_v4_scale_dynamic_top_n(dynamic_top_n: dict, available_names: int) -> dict:
    if available_names >= max(dynamic_top_n.values()):
        return dict(dynamic_top_n)
    return {
        "high": max(1, min(int(dynamic_top_n.get("high", 15)), available_names)),
        "normal": max(1, min(int(dynamic_top_n.get("normal", 28)), available_names)),
        "low": max(1, min(int(dynamic_top_n.get("low", 45)), available_names)),
    }


def build_confidence_portfolio(
    predictions: pd.DataFrame,
    *,
    strategy_name: str,
    dataset_name: str,
    mode: str = "dynamic",
    top_k: int | None = None,
    dynamic_top_n: dict = DYNAMIC_TOP_N,
    confidence_threshold: float = CONFIDENCE_THRESHOLD,
    high_confidence_threshold: float = HIGH_CONFIDENCE_THRESHOLD,
    low_confidence_threshold: float = LOW_CONFIDENCE_THRESHOLD,
    spread_threshold: float = SPREAD_THRESHOLD,
    risk_quality_threshold: float = RISK_QUALITY_THRESHOLD,
    max_weight: float = MAX_WEIGHT,
    defensive_max_weight: float = DEFENSIVE_MAX_WEIGHT,
    risk_on_max_weight: float = RISK_ON_MAX_WEIGHT,
    min_active_names: int = MIN_ACTIVE_NAMES,
    no_trade_band: float = NO_TRADE_BAND,
    softmax_temperature: float = SOFTMAX_TEMPERATURE,
    defensive_temperature: float = DEFENSIVE_TEMPERATURE,
    risk_aversion: float = RISK_AVERSION,
    downside_aversion: float = DOWNSIDE_AVERSION,
    tail_risk_aversion: float = TAIL_RISK_AVERSION,
    loser_rank_cutoff: float = LOSER_RANK_CUTOFF,
    bottom_prob_cutoff: float = BOTTOM_PROB_CUTOFF,
) -> tuple[PortfolioWeights, pd.DataFrame]:
    validated = validate_prediction_frame(predictions)
    tickers = sorted(validated["ticker"].unique().tolist())
    available_names = len(tickers)
    effective_dynamic_top_n = _minho_v4_scale_dynamic_top_n(dynamic_top_n, available_names)
    effective_min_active = max(1, min(int(min_active_names), available_names))
    weights = pd.DataFrame(0.0, index=pd.DatetimeIndex(sorted(validated["date"].unique())), columns=tickers)
    weights.index.name = "date"
    diagnostics = []
    previous_row = None

    for date_value, frame_date in validated.groupby("date", sort=True):
        frame_date = frame_date.copy()
        n = len(frame_date)
        frame_date["confidence"] = frame_date.get("confidence", 0.5)
        frame_date["confidence"] = frame_date["confidence"].fillna(0.5).clip(0.0, 1.0)
        frame_date["bottom_probability"] = frame_date.get("bottom_probability", 0.2)
        frame_date["bottom_probability"] = frame_date["bottom_probability"].fillna(0.2).clip(0.0, 1.0)
        frame_date["expected_volatility"] = frame_date.get("expected_volatility", 0.0)
        frame_date["expected_downside"] = frame_date.get("expected_downside", 0.0)
        frame_date["expected_worst_path"] = frame_date.get("expected_worst_path", 0.0)
        frame_date[["expected_volatility", "expected_downside", "expected_worst_path"]] = frame_date[[
            "expected_volatility", "expected_downside", "expected_worst_path"
        ]].fillna(0.0)
        frame_date["alpha_rank"] = _rank01(frame_date["expected_return"], ascending=True)
        aux_cols = [c for c in frame_date.columns if c.startswith("expected_alpha_") and c.endswith("d")]
        aux_rank = pd.concat([_rank01(frame_date[c], ascending=True) for c in aux_cols], axis=1).mean(axis=1) if aux_cols else frame_date["alpha_rank"]
        frame_date["multi_horizon_rank"] = 0.70 * frame_date["alpha_rank"] + 0.30 * aux_rank
        tail_risk = (-frame_date["expected_worst_path"]).clip(lower=0.0)
        frame_date["raw_risk_load"] = (
            risk_aversion * frame_date["expected_volatility"].clip(lower=0.0)
            + downside_aversion * frame_date["expected_downside"].clip(lower=0.0)
            + tail_risk_aversion * tail_risk
        )
        frame_date["risk_quality"] = _rank01(-frame_date["raw_risk_load"], ascending=True)
        frame_date["risk_penalty"] = np.exp(-frame_date["raw_risk_load"].clip(lower=0.0))
        frame_date["portfolio_score"] = (
            frame_date["multi_horizon_rank"]
            * (0.35 + 0.85 * frame_date["confidence"])
            * frame_date["risk_penalty"]
            - 0.35 * frame_date["bottom_probability"]
        )
        spread = float(frame_date["expected_return"].quantile(0.90) - frame_date["expected_return"].quantile(0.10))
        top_pool = frame_date.nlargest(max(1, int(np.ceil(n * 0.20))), "expected_return")
        top_conf = float(top_pool["confidence"].mean())
        top_risk_quality = float(top_pool["risk_quality"].mean())
        risk_on = top_conf >= high_confidence_threshold and spread >= spread_threshold and top_risk_quality >= risk_quality_threshold
        uncertain = top_conf < low_confidence_threshold or spread < spread_threshold * 0.50
        tradable_pool = frame_date[
            (frame_date["alpha_rank"] > loser_rank_cutoff)
            & (frame_date["bottom_probability"] < bottom_prob_cutoff)
        ].copy()
        if len(tradable_pool) < effective_min_active:
            tradable_pool = frame_date.nlargest(min(n, max(effective_min_active, effective_dynamic_top_n.get("normal", effective_min_active))), "portfolio_score").copy()

        if mode == "top_k":
            regime = f"top_{top_k}"
            k = min(len(tradable_pool), max(1, int(top_k or len(tradable_pool))))
            row_cap = max(max_weight, 1.0 / k + 1e-12)
            temperature = softmax_temperature
            concentration_on = True
        elif mode == "broad":
            regime = "low"
            k = min(len(tradable_pool), max(effective_min_active, effective_dynamic_top_n.get("low", len(tradable_pool))))
            row_cap = max(defensive_max_weight, 1.0 / k + 1e-12)
            temperature = defensive_temperature
            concentration_on = False
        else:
            if risk_on:
                regime = "high"
                k = min(len(tradable_pool), max(effective_min_active, effective_dynamic_top_n["high"]))
                row_cap = max(risk_on_max_weight, 1.0 / k + 1e-12)
                temperature = softmax_temperature
                concentration_on = True
            elif uncertain:
                regime = "low"
                k = min(len(tradable_pool), max(effective_min_active, effective_dynamic_top_n["low"]))
                row_cap = max(defensive_max_weight, 1.0 / k + 1e-12)
                temperature = defensive_temperature
                concentration_on = False
            else:
                regime = "normal"
                k = min(len(tradable_pool), max(effective_min_active, effective_dynamic_top_n["normal"]))
                row_cap = max(max_weight, 1.0 / k + 1e-12)
                temperature = softmax_temperature
                concentration_on = top_conf >= confidence_threshold

        chosen = tradable_pool.nlargest(k, "portfolio_score").copy()
        raw = _softmax_weights(chosen.set_index("ticker")["portfolio_score"], temperature)
        row = pd.Series(0.0, index=tickers)
        row.loc[raw.index] = _project_capped_simplex(raw, row_cap)
        row = _apply_no_trade_band(row, previous_row, no_trade_band)
        row = _project_capped_simplex(row, row_cap)
        weights.loc[pd.Timestamp(date_value), row.index] = row.values
        previous_row = row
        diagnostics.append({
            "date": pd.Timestamp(date_value),
            "mode": mode,
            "regime": regime,
            "candidate_names_after_filter": int(len(tradable_pool)),
            "selected_names": int((row > 0).sum()),
            "effective_names": _effective_names(row),
            "max_weight": float(row.max()),
            "effective_max_weight_cap": float(row_cap),
            "top_confidence": top_conf,
            "top_risk_quality": top_risk_quality,
            "predicted_spread_90_10": spread,
            "concentration_on": bool(concentration_on),
            "risk_on_gate": bool(risk_on),
            "uncertain_gate": bool(uncertain),
            "avg_bottom_probability_selected": float(chosen["bottom_probability"].mean()),
            "avg_risk_quality_selected": float(chosen["risk_quality"].mean()),
        })

    clean_weights = validate_weights_frame(weights)
    return PortfolioWeights(
        weights=clean_weights,
        dataset_name=dataset_name,
        strategy_name=strategy_name,
        metadata={
            "mode": mode,
            "dynamic_top_n_original": dynamic_top_n,
            "dynamic_top_n_effective": effective_dynamic_top_n,
            "max_weight": max_weight,
            "defensive_max_weight": defensive_max_weight,
            "risk_on_max_weight": risk_on_max_weight,
            "min_active_names_original": min_active_names,
            "min_active_names_effective": effective_min_active,
            "no_trade_band": no_trade_band,
            "confidence_threshold": confidence_threshold,
            "high_confidence_threshold": high_confidence_threshold,
            "low_confidence_threshold": low_confidence_threshold,
            "spread_threshold": spread_threshold,
            "risk_quality_threshold": risk_quality_threshold,
            "risk_aversion": risk_aversion,
            "downside_aversion": downside_aversion,
            "tail_risk_aversion": tail_risk_aversion,
            "loser_rank_cutoff": loser_rank_cutoff,
            "bottom_prob_cutoff": bottom_prob_cutoff,
        },
    ), pd.DataFrame(diagnostics)


def _run_minho_v4_regime_backtest(dataset_name: str) -> dict[str, object]:
    spec_obj = get_dataset_spec(dataset_name, repo_root=repo_root)
    benchmark = spec_obj.benchmark_ticker.upper()
    tradable = sorted([ticker.upper() for ticker in spec_obj.tickers if ticker.upper() not in {benchmark, "SPY"}])
    if not tradable:
        raise ValueError(f"{dataset_name} has no tradable tickers after excluding benchmark/SPY context.")

    loaded_prices = load_prices(spec_obj, repo_root=repo_root).copy()
    loaded_prices["date"] = pd.to_datetime(loaded_prices["date"], utc=True).dt.tz_localize(None)
    loaded_prices["ticker"] = loaded_prices["ticker"].astype(str).str.upper()
    available_tradable = sorted(set(tradable).intersection(set(loaded_prices["ticker"])))
    feature_prices = _minho_v4_prices_with_spy_context(loaded_prices, spec_obj)

    all_predictions = predict_from_prices(
        model=minho_v4_model,
        prices=feature_prices,
        train_mean=MINHO_V4_TRAIN_MEAN,
        train_std=MINHO_V4_TRAIN_STD,
        feature_names=MINHO_V4_FEATURE_NAMES,
        seq_len=MINHO_V4_SEQ_LEN,
        horizon=MINHO_V4_HORIZON,
        tickers=available_tradable,
    )
    all_predictions = all_predictions.loc[
        (all_predictions["date"] >= pd.Timestamp(spec_obj.test_start))
        & (all_predictions["date"] <= pd.Timestamp(spec_obj.test_end))
    ].reset_index(drop=True)
    if all_predictions.empty:
        raise ValueError(f"No predictions produced for {dataset_name} test window.")
    rebalance_dates = pd.DatetimeIndex(sorted(all_predictions["date"].unique()))[::MINHO_V4_REBALANCE_EVERY_N_DAYS]
    predictions = all_predictions.loc[all_predictions["date"].isin(rebalance_dates)].reset_index(drop=True)
    predictions = validate_prediction_frame(predictions, dataset_name=spec_obj, horizon=MINHO_V4_HORIZON, repo_root=repo_root)

    strategy_name = f"minho_v4_dynamic_risk_{dataset_name}"
    portfolio, diagnostics = build_confidence_portfolio(
        predictions,
        strategy_name=strategy_name,
        dataset_name=spec_obj.identifier,
        mode="dynamic",
    )
    portfolio.metadata.update({
        "model_name": MINHO_V4_MODEL_NAME,
        "source_run_id": MINHO_V4_SOURCE_RUN_ID,
        "source_run_name": MINHO_V4_SOURCE_RUN_NAME,
        "rebalance_frequency": MINHO_V4_REBALANCE_FREQUENCY,
        "benchmark_ticker": benchmark,
        "spy_feature_context_only": True,
    })
    portfolio = PortfolioWeights(
        weights=validate_weights_frame(portfolio.weights, dataset_name=spec_obj, repo_root=repo_root),
        dataset_name=spec_obj.identifier,
        strategy_name=strategy_name,
        metadata=portfolio.metadata,
    )
    result = backtest_weights(spec_obj, portfolio, benchmark=benchmark, repo_root=repo_root)

    dataset_output_dir = MINHO_V4_REGIME_BACKTEST_OUTPUT_DIR / dataset_name
    dataset_output_dir.mkdir(parents=True, exist_ok=True)
    artifact_paths = write_backtest_artifacts(result, dataset_output_dir)
    predictions_path = dataset_output_dir / "predictions.parquet"
    diagnostics_path = dataset_output_dir / "portfolio_diagnostics.parquet"
    metadata_path = dataset_output_dir / "metadata.json"
    predictions.to_parquet(predictions_path, index=False)
    diagnostics.to_parquet(diagnostics_path, index=False)
    metadata = {
        "dataset_name": dataset_name,
        "source_run_id": MINHO_V4_SOURCE_RUN_ID,
        "source_run_name": MINHO_V4_SOURCE_RUN_NAME,
        "model_name": MINHO_V4_MODEL_NAME,
        "benchmark_ticker": benchmark,
        "test_start": str(spec_obj.test_start),
        "test_end": str(spec_obj.test_end),
        "horizon": MINHO_V4_HORIZON,
        "rebalance_frequency": MINHO_V4_REBALANCE_FREQUENCY,
        "active_ticker_count": len(available_tradable),
        "portfolio_metadata": portfolio.metadata,
    }
    metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")
    artifact_paths.update({"predictions": str(predictions_path), "portfolio_diagnostics": str(diagnostics_path), "metadata": str(metadata_path)})

    print(
        f"{dataset_name}: {predictions['date'].nunique()} rebalance dates, "
        f"{len(available_tradable)} tradable tickers, total return {result.metrics.get('total_return', np.nan):.2%}"
    )
    return {
        "dataset_name": dataset_name,
        "spec": spec_obj,
        "predictions": predictions,
        "portfolio": portfolio,
        "diagnostics": diagnostics,
        "result": result,
        "artifact_paths": artifact_paths,
        "rebalance_count": int(predictions["date"].nunique()),
        "active_ticker_count": int(len(available_tradable)),
    }

print("Minho v4 portfolio and backtest helpers are ready.")


Minho v4 portfolio and backtest helpers are ready.


In [4]:
minho_v4_regime_backtest_runs = {}
for dataset_name in MINHO_V4_REGIME_BACKTEST_DATASETS:
    minho_v4_regime_backtest_runs[dataset_name] = _run_minho_v4_regime_backtest(dataset_name)

print("Completed Minho v4 four-regime backtests:", list(minho_v4_regime_backtest_runs))


regime_modern_tech_gain_2022_2026: 220 rebalance dates, 40 tradable tickers, total return 129.23%
regime_financial_crisis_loss_2005_2010: 303 rebalance dates, 33 tradable tickers, total return 50.76%
regime_nineties_volatility_1995_1999: 236 rebalance dates, 41 tradable tickers, total return 440.58%
regime_oil_pre2014_energy_2010_2013: 202 rebalance dates, 22 tradable tickers, total return 78.81%
Completed Minho v4 four-regime backtests: ['regime_modern_tech_gain_2022_2026', 'regime_financial_crisis_loss_2005_2010', 'regime_nineties_volatility_1995_1999', 'regime_oil_pre2014_energy_2010_2013']


In [5]:
minho_v4_summary_rows = []
for dataset_name, payload in minho_v4_regime_backtest_runs.items():
    spec_obj = payload["spec"]
    result_obj = payload["result"]
    diagnostics = payload["diagnostics"]
    metrics_obj = dict(result_obj.metrics)
    total_return = float(metrics_obj.get("total_return", np.nan))
    ending_bankroll = MINHO_V4_REGIME_BACKTEST_ALLOCATION * (1.0 + total_return)
    minho_v4_summary_rows.append(
        {
            "dataset_name": dataset_name,
            "benchmark_ticker": spec_obj.benchmark_ticker,
            "test_start": spec_obj.test_start,
            "test_end": spec_obj.test_end,
            "allocation": MINHO_V4_REGIME_BACKTEST_ALLOCATION,
            "ending_bankroll": ending_bankroll,
            "profit_loss": ending_bankroll - MINHO_V4_REGIME_BACKTEST_ALLOCATION,
            "total_return": total_return,
            "annual_return": metrics_obj.get("annual_return", np.nan),
            "annual_volatility": metrics_obj.get("annual_volatility", np.nan),
            "sharpe": metrics_obj.get("sharpe", np.nan),
            "sortino": metrics_obj.get("sortino", np.nan),
            "max_drawdown": metrics_obj.get("max_drawdown", np.nan),
            "average_turnover": metrics_obj.get("average_turnover", np.nan),
            "benchmark_total_return": metrics_obj.get("benchmark_total_return", np.nan),
            "excess_return_vs_benchmark": metrics_obj.get("excess_return_vs_benchmark", np.nan),
            "rebalance_count": payload["rebalance_count"],
            "active_ticker_count": payload["active_ticker_count"],
            "avg_selected_names": float(diagnostics["selected_names"].mean()) if not diagnostics.empty else np.nan,
            "avg_effective_names": float(diagnostics["effective_names"].mean()) if not diagnostics.empty else np.nan,
            "avg_max_weight": float(diagnostics["max_weight"].mean()) if not diagnostics.empty else np.nan,
            "max_single_weight": float(payload["result"].weights.max(axis=1).max()),
            "risk_on_day_share": float(diagnostics["risk_on_gate"].mean()) if not diagnostics.empty else np.nan,
            "uncertain_day_share": float(diagnostics["uncertain_gate"].mean()) if not diagnostics.empty else np.nan,
        }
    )

minho_v4_regime_backtest_summary = pd.DataFrame(minho_v4_summary_rows)
minho_v4_combined_ending_bankroll = float(minho_v4_regime_backtest_summary["ending_bankroll"].sum())
minho_v4_combined_profit_loss = minho_v4_combined_ending_bankroll - MINHO_V4_REGIME_BACKTEST_TOTAL_BANKROLL
minho_v4_combined_total_return = minho_v4_combined_profit_loss / MINHO_V4_REGIME_BACKTEST_TOTAL_BANKROLL
minho_v4_combined_row = pd.DataFrame(
    [
        {
            "dataset_name": "COMBINED_4_REGIME_BANKROLL",
            "benchmark_ticker": "mixed",
            "test_start": min(minho_v4_regime_backtest_summary["test_start"]),
            "test_end": max(minho_v4_regime_backtest_summary["test_end"]),
            "allocation": MINHO_V4_REGIME_BACKTEST_TOTAL_BANKROLL,
            "ending_bankroll": minho_v4_combined_ending_bankroll,
            "profit_loss": minho_v4_combined_profit_loss,
            "total_return": minho_v4_combined_total_return,
            "annual_return": np.nan,
            "annual_volatility": np.nan,
            "sharpe": np.nan,
            "sortino": np.nan,
            "max_drawdown": np.nan,
            "average_turnover": np.nan,
            "benchmark_total_return": np.nan,
            "excess_return_vs_benchmark": np.nan,
            "rebalance_count": int(minho_v4_regime_backtest_summary["rebalance_count"].sum()),
            "active_ticker_count": int(minho_v4_regime_backtest_summary["active_ticker_count"].sum()),
            "avg_selected_names": np.nan,
            "avg_effective_names": np.nan,
            "avg_max_weight": np.nan,
            "max_single_weight": np.nan,
            "risk_on_day_share": np.nan,
            "uncertain_day_share": np.nan,
        }
    ]
)
minho_v4_regime_backtest_summary = pd.concat([minho_v4_regime_backtest_summary, minho_v4_combined_row], ignore_index=True)
summary_csv_path = MINHO_V4_REGIME_BACKTEST_OUTPUT_DIR / "minho_v4_four_regime_bankroll_summary.csv"
summary_parquet_path = MINHO_V4_REGIME_BACKTEST_OUTPUT_DIR / "minho_v4_four_regime_bankroll_summary.parquet"
minho_v4_regime_backtest_summary.to_csv(summary_csv_path, index=False)
minho_v4_regime_backtest_summary.to_parquet(summary_parquet_path, index=False)

print(f"Combined ending bankroll: ${minho_v4_combined_ending_bankroll:,.2f}")
print(f"Combined profit/loss: ${minho_v4_combined_profit_loss:,.2f}")
print(f"Combined total return: {minho_v4_combined_total_return:.2%}")
display(minho_v4_regime_backtest_summary)


Combined ending bankroll: $13,192,489.31
Combined profit/loss: $8,392,489.31
Combined total return: 174.84%


,dataset_name,benchmark_ticker,test_start,test_end,allocation,ending_bankroll,profit_loss,total_return,annual_return,annual_volatility,...,benchmark_total_return,excess_return_vs_benchmark,rebalance_count,active_ticker_count,avg_selected_names,avg_effective_names,avg_max_weight,max_single_weight,risk_on_day_share,uncertain_day_share
0,regime_modern_tech_gain_2022_2026,XLK,2022-01-03,2026-05-21,1200000.0,2.750763e+06,1.550763e+06,1.292303,0.208634,0.344445,...,1.100637,0.191666,220,40,28.372727,27.736934,0.039816,0.066667,0.063636,0.150000
1,regime_financial_crisis_loss_2005_2010,XLF,2005-01-03,2010-12-31,1200000.0,1.809083e+06,6.090826e+05,0.507569,0.070928,0.491918,...,-0.392930,0.900499,303,33,27.369637,26.826291,0.040446,0.066667,0.089109,0.293729
2,regime_nineties_volatility_1995_1999,MDY,1995-05-05,1999-12-31,1200000.0,6.486942e+06,5.286942e+06,4.405785,0.436694,0.198828,...,1.407843,2.997943,236,41,27.898305,27.203174,0.040602,0.066667,0.063559,0.080508
3,regime_oil_pre2014_energy_2010_2013,XOP,2010-01-04,2013-12-31,1200000.0,2.145701e+06,9.457013e+05,0.788084,0.156832,0.260299,...,0.645439,0.142645,202,22,19.861386,19.861386,0.050452,0.066667,0.024752,0.430693
4,COMBINED_4_REGIME_BANKROLL,mixed,1995-05-05,2026-05-21,4800000.0,1.319249e+07,8.392489e+06,1.748435,NaN,NaN,...,NaN,NaN,961,136,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
MINHO_V4_FOUR_REGIME_LOG_TO_MLFLOW = os.environ.get("MINHO_V4_FOUR_REGIME_LOG_TO_MLFLOW", "1").strip() not in {"0", "false", "False", "no"}

if MINHO_V4_FOUR_REGIME_LOG_TO_MLFLOW:
    init_mlflow(repo_root=repo_root)
    for dataset_name, payload in minho_v4_regime_backtest_runs.items():
        spec_obj = payload["spec"]
        result_obj = payload["result"]
        portfolio_obj = payload["portfolio"]
        predictions_obj = payload["predictions"]
        diagnostics_obj = payload["diagnostics"]
        bankroll_row = minho_v4_regime_backtest_summary.loc[minho_v4_regime_backtest_summary["dataset_name"] == dataset_name].iloc[0]
        with start_run(
            run_name=f"{MINHO_V4_MODEL_NAME}_four_regime_{dataset_name}",
            dataset_name=spec_obj,
            tags={
                "workflow": "minho_v4_dynamic_risk_four_regime_proxy_backtest",
                "model_family": "torch",
                "model_name": MINHO_V4_MODEL_NAME,
                "source_author": "Minho",
                "source_run_id": MINHO_V4_SOURCE_RUN_ID,
                "source_shared_run_name": MINHO_V4_SOURCE_RUN_NAME,
                "prediction_horizon": str(MINHO_V4_HORIZON),
                "rebalance_frequency": MINHO_V4_REBALANCE_FREQUENCY,
                "bankroll_allocation": str(MINHO_V4_REGIME_BACKTEST_ALLOCATION),
            },
            repo_root=repo_root,
        ):
            mlflow.log_params({
                "model_name": MINHO_V4_MODEL_NAME,
                "source_run_id": MINHO_V4_SOURCE_RUN_ID,
                "source_run_name": MINHO_V4_SOURCE_RUN_NAME,
                "checkpoint_path": str(MINHO_V4_CHECKPOINT_PATH.relative_to(repo_root)),
                "source_notebook_path": str(MINHO_V4_NOTEBOOK_PATH.relative_to(repo_root)),
                "dataset_name": dataset_name,
                "benchmark_ticker": spec_obj.benchmark_ticker,
                "test_start": str(spec_obj.test_start),
                "test_end": str(spec_obj.test_end),
                "horizon": MINHO_V4_HORIZON,
                "seq_len": MINHO_V4_SEQ_LEN,
                "feature_count": len(MINHO_V4_FEATURE_NAMES),
                "feature_list": ",".join(MINHO_V4_FEATURE_NAMES),
                "rebalance_frequency": MINHO_V4_REBALANCE_FREQUENCY,
                "portfolio_builder": "build_confidence_portfolio",
                "active_ticker_count": payload["active_ticker_count"],
                "bankroll_allocation": MINHO_V4_REGIME_BACKTEST_ALLOCATION,
            })
            mlflow.log_metrics({
                "bankroll_allocation": float(bankroll_row["allocation"]),
                "ending_bankroll": float(bankroll_row["ending_bankroll"]),
                "profit_loss": float(bankroll_row["profit_loss"]),
                "avg_selected_names": float(bankroll_row["avg_selected_names"]),
                "avg_effective_names": float(bankroll_row["avg_effective_names"]),
                "avg_max_weight": float(bankroll_row["avg_max_weight"]),
                "max_single_weight": float(bankroll_row["max_single_weight"]),
                "risk_on_day_share": float(bankroll_row["risk_on_day_share"]),
                "uncertain_day_share": float(bankroll_row["uncertain_day_share"]),
                "combined_ending_bankroll": minho_v4_combined_ending_bankroll,
                "combined_profit_loss": minho_v4_combined_profit_loss,
                "combined_total_return": minho_v4_combined_total_return,
            })
            log_predictions(predictions_obj)
            log_portfolio(portfolio_obj)
            log_backtest(result_obj)
            for artifact_name in ["predictions", "portfolio_diagnostics", "metadata"]:
                artifact_path = Path(payload["artifact_paths"].get(artifact_name, ""))
                if artifact_path.exists():
                    mlflow.log_artifact(str(artifact_path), artifact_path="four_regime_artifacts")
            for artifact_path in [summary_csv_path, summary_parquet_path, MINHO_V4_MANIFEST_PATH]:
                if artifact_path.exists():
                    mlflow.log_artifact(str(artifact_path), artifact_path="four_regime_artifacts")
            log_model_submission(
                {"lstm_checkpoint": MINHO_V4_CHECKPOINT_PATH},
                model_name=MINHO_V4_MODEL_NAME,
                model_family="torch",
                feature_names=MINHO_V4_FEATURE_NAMES,
                target=MINHO_V4_MANIFEST.get("target", MINHO_V4_CHECKPOINT.get("data", {}).get("target", "forward_alpha_5d_vs_spy")),
                horizon=MINHO_V4_HORIZON,
                rebalance_frequency=MINHO_V4_REBALANCE_FREQUENCY,
                preprocessing={
                    "scaler": "train_mean_std_from_downloaded_checkpoint",
                    "checkpoint_preprocessing_keys": sorted(MINHO_V4_PREPROCESSING.keys()),
                    "spy_feature_context_only": True,
                },
                model_config={
                    **MINHO_V4_CHECKPOINT.get("model_config", {}),
                    "portfolio_builder": "build_confidence_portfolio",
                    "portfolio_config": MINHO_V4_PORTFOLIO_CONFIG,
                    "required_functions": ["build_model_features", "predict_from_prices", "build_confidence_portfolio"],
                },
                source_files=[MINHO_V4_NOTEBOOK_PATH],
                notes="Minho v4 dynamic-risk LSTM checkpoint rerun on four proxy-regime datasets with SPY feature context.",
            )
    print("Logged Minho v4 four-regime backtests to MLflow.")
else:
    print("Skipping MLflow logging because MINHO_V4_FOUR_REGIME_LOG_TO_MLFLOW is disabled.")


🏃 View run Minho_RankingConfidence_LSTM_v4_dynamic_risk_four_regime_regime_modern_tech_gain_2022_2026 at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/10/runs/fdeb79932e9f4bbabbb812fd7464edd3
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/10
🏃 View run Minho_RankingConfidence_LSTM_v4_dynamic_risk_four_regime_regime_financial_crisis_loss_2005_2010 at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/11/runs/14e1842d33f24e488c9db99e7fedc4d7
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/11
🏃 View run Minho_RankingConfidence_LSTM_v4_dynamic_risk_four_regime_regime_nineties_volatility_1995_1999 at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/12/runs/df32d6371a53401da439814197da5fcc
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/12
🏃 View run Minho_RankingConfidence_LSTM_v4_dynamic_risk_four_regime_regime_oil_pre2014_energy_2010_2013 at: https://adams-macboo